In [1]:
# =============================================================================
# CELL 1: Imports, Configuration, Embedding Model, ChromaDB Connection
# =============================================================================
# Consolidated setup for the collaborative discussion agent.
# Merges Notebook 1 Cells 1 (imports/config), 5 (embeddings), 7 (ChromaDB init).
# Removed: Whisper, pyttsx3, Audio — this notebook is text-only.
# Added: Discussion agent constants (trigger detection, throttling, reformulation).

# --- Standard library ---
import os
import re
import json
import uuid
import time
from pathlib import Path

# --- HTTP / API communication ---
import requests

# --- PDF text extraction ---
from PyPDF2 import PdfReader

# --- Embeddings ---
from sentence_transformers import SentenceTransformer

# --- Vector store ---
import chromadb
from chromadb.config import Settings

# --- Numerical ---
import numpy as np
from numpy import dot
from numpy.linalg import norm

# =============================================================================
# CONFIGURATION
# =============================================================================

# -- Ollama --
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL = "mistral"

# -- Embedding model --
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"
EMBEDDING_DIMENSIONS = 384
BGE_QUERY_PREFIX = "Represent this sentence: "

# -- Chunking --
CHUNK_SIZE = 800
CHUNK_OVERLAP = 100
CHUNK_MIN_WORDS = 20
CHUNK_MIN_LETTER_RATIO = 0.40

# -- Retrieval --
TOP_K = 5
CHROMA_COLLECTION = "rag_documents"
CHROMA_PERSIST_DIR = "./chroma_data"

# -- Document classification --
ACADEMIC_CONTENT_MARKERS = [
    "abstract", "introduction", "background", "chapter 1",
    "1. introduction", "1 introduction", "literature review"
]
ACADEMIC_SIGNALS = [
    "thesis", "dissertation", "submitted in partial fulfillment",
    "degree of", "bachelor", "master", "doctor of philosophy",
    "university", "department of", "supervisor", "declaration",
    "certificate", "acknowledgement", "acknowledgment"
]
ACADEMIC_SIGNAL_THRESHOLD = 2

# -- Conversation history --
MAX_HISTORY_TURNS = 5

# -- Discussion agent: Trigger detection --
# The classifier prompt is kept short to minimize latency.
# TRIGGER_CONTEXT_WINDOW controls how many recent messages the classifier sees
# alongside the new message to make its decision.
TRIGGER_CONTEXT_WINDOW = 3

# -- Discussion agent: Response throttling --
# MIN_RESPONSE_GAP: minimum number of user messages between bot responses.
#   Prevents the bot from responding to consecutive messages.
# MAX_BOT_RATIO: if the bot has authored more than this fraction of the last
#   RATIO_WINDOW messages, it backs off even if a trigger fires.
# RATIO_WINDOW: how many recent messages to consider for the ratio check.
MIN_RESPONSE_GAP = 2
MAX_BOT_RATIO = 0.3
RATIO_WINDOW = 10

# -- Discussion agent: Query reformulation --
# REFORMULATION_CONTEXT_TURNS: how many recent conversation turns to include
# when asking the LLM to reformulate a vague query into a retrieval-ready one.
REFORMULATION_CONTEXT_TURNS = 3

# -- File paths --
UPLOAD_DIR = "./uploads"
os.makedirs(UPLOAD_DIR, exist_ok=True)
os.makedirs(CHROMA_PERSIST_DIR, exist_ok=True)

# =============================================================================
# HELPER: Cosine similarity (used by boilerplate scoring and retrieval)
# =============================================================================
def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    return dot(a, b) / (norm(a) * norm(b))

# =============================================================================
# LOAD EMBEDDING MODEL
# =============================================================================
print(f"Loading embedding model: {EMBEDDING_MODEL_NAME}")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

test_embedding = embedding_model.encode("Dimension check.")
assert test_embedding.shape[0] == EMBEDDING_DIMENSIONS, \
    f"Expected {EMBEDDING_DIMENSIONS}d, got {test_embedding.shape[0]}d"
print(f"✓ Embedding model loaded ({EMBEDDING_DIMENSIONS}d)\n")

# =============================================================================
# CONNECT TO CHROMADB
# =============================================================================
chroma_client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)
print(f"✓ ChromaDB connected at {CHROMA_PERSIST_DIR}")

# Check if we already have an ingested collection (from a previous run)
try:
    existing = chroma_client.get_collection(name=CHROMA_COLLECTION)
    print(f"  Existing collection '{CHROMA_COLLECTION}': {existing.count()} chunks")
except Exception:
    print(f"  No existing collection — run Cell 2 to ingest a document")

# =============================================================================
# OLLAMA CONNECTIVITY CHECK
# =============================================================================
try:
    response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
    models = [m["name"] for m in response.json().get("models", [])]
    print(f"\n✓ Ollama reachable")
    print(f"  Available models: {', '.join(models) if models else 'none found'}")
    if not any(OLLAMA_MODEL in m for m in models):
        print(f"  ⚠ '{OLLAMA_MODEL}' not found — run: ollama pull {OLLAMA_MODEL}")
except requests.ConnectionError:
    print(f"\n⚠ Ollama not reachable at {OLLAMA_BASE_URL} — make sure it's running")

# =============================================================================
# SUMMARY
# =============================================================================
print(f"\n{'=' * 60}")
print("CONFIGURATION SUMMARY")
print(f"{'=' * 60}")
print(f"  LLM:               {OLLAMA_MODEL} @ {OLLAMA_BASE_URL}")
print(f"  Embeddings:        {EMBEDDING_MODEL_NAME} ({EMBEDDING_DIMENSIONS}d)")
print(f"  Chunks:            {CHUNK_SIZE} chars, {CHUNK_OVERLAP} overlap")
print(f"  Retrieval:         top-{TOP_K}, boilerplate-penalized")
print(f"  History:           last {MAX_HISTORY_TURNS} turns")
print(f"  Trigger context:   last {TRIGGER_CONTEXT_WINDOW} messages")
print(f"  Throttling:        min {MIN_RESPONSE_GAP} msg gap, max {MAX_BOT_RATIO:.0%} bot ratio (window={RATIO_WINDOW})")
print(f"  Query reformulate: last {REFORMULATION_CONTEXT_TURNS} turns")
print(f"\n✓ Cell 1 complete — ready for Cell 2 (ingestion pipeline)")

c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading embedding model: BAAI/bge-small-en-v1.5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 701.20it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Embedding model loaded (384d)

✓ ChromaDB connected at ./chroma_data
  Existing collection 'rag_documents': 82 chunks

✓ Ollama reachable
  Available models: mistral:latest

CONFIGURATION SUMMARY
  LLM:               mistral @ http://localhost:11434
  Embeddings:        BAAI/bge-small-en-v1.5 (384d)
  Chunks:            800 chars, 100 overlap
  Retrieval:         top-5, boilerplate-penalized
  History:           last 5 turns
  Trigger context:   last 3 messages
  Throttling:        min 2 msg gap, max 30% bot ratio (window=10)
  Query reformulate: last 3 turns

✓ Cell 1 complete — ready for Cell 2 (ingestion pipeline)


In [2]:
# =============================================================================
# CELL 2: Document Upload and Ingestion Pipeline
# =============================================================================
# Consolidates Notebook 1 Cells 3 (load), 4 (chunk), 6 (classify/score),
# and 7 (ingest) into a single cell with one top-level entry point:
#   upload_and_ingest(file_path) → collection
#
# All functions are unchanged from Notebook 1 — just reorganized.
# The only new addition is the wrapper function at the bottom.

# =============================================================================
# DOCUMENT LOADING (from Notebook 1, Cell 3)
# =============================================================================

def extract_text_from_pdf(file_path: str) -> str:
    """Extract all text from a PDF file, page by page."""
    reader = PdfReader(file_path)
    pages_text = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            pages_text.append(text)
        else:
            print(f"  ⚠ Page {i+1}: no text extracted (image or blank)")
    return "\n\n".join(pages_text)


def load_document(file_path: str) -> dict:
    """
    Load a PDF and return extracted text with metadata.
    Returns: {"doc_id": str, "filename": str, "text": str}
    """
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")
    if path.suffix.lower() != ".pdf":
        raise ValueError(f"Unsupported format: {path.suffix}. Only .pdf is supported.")

    print(f"  Loading PDF: {path.name}")
    text = extract_text_from_pdf(file_path)
    doc_id = uuid.uuid4().hex[:12]

    return {"doc_id": doc_id, "filename": path.name, "text": text}


# =============================================================================
# TEXT CHUNKING WITH QUALITY FILTER (from Notebook 1, Cell 4)
# =============================================================================

def split_oversized_sentence(sentence: str, chunk_size: int) -> list[str]:
    """Split a sentence exceeding chunk_size on clause boundaries."""
    if len(sentence) <= chunk_size:
        return [sentence]

    split_points = []
    for pattern in ["; ", ", ", " - ", " — "]:
        for match in re.finditer(re.escape(pattern), sentence):
            split_points.append(match.start() + len(pattern))

    if split_points:
        mid = len(sentence) // 2
        best = min(split_points, key=lambda x: abs(x - mid))
        left = sentence[:best].strip()
        right = sentence[best:].strip()
    else:
        mid = len(sentence) // 2
        space_pos = sentence.rfind(" ", 0, mid + 50)
        if space_pos == -1:
            space_pos = mid
        left = sentence[:space_pos].strip()
        right = sentence[space_pos:].strip()

    return split_oversized_sentence(left, chunk_size) + \
           split_oversized_sentence(right, chunk_size)


def is_quality_chunk(chunk: str) -> bool:
    """Filter out low-content chunks (title pages, signature lines, etc.)."""
    words = [w for w in chunk.split() if len(w) > 1 and any(c.isalpha() for c in w)]
    if len(words) < CHUNK_MIN_WORDS:
        return False
    if len(chunk) == 0:
        return False
    letters = sum(1 for c in chunk if c.isalpha())
    if letters / len(chunk) < CHUNK_MIN_LETTER_RATIO:
        return False
    return True


def chunk_text(
    text: str,
    chunk_size: int = CHUNK_SIZE,
    overlap: int = CHUNK_OVERLAP,
    apply_quality_filter: bool = True
) -> list[str]:
    """
    Split text into overlapping, sentence-aware chunks.
    Quality filter removes low-content chunks automatically.
    """
    # Split into sentences
    sentences = re.split(r'(?<=[.!?])\s+|\n\n+', text)
    sentences = [s.strip() for s in sentences if s.strip()]

    # Handle oversized sentences
    processed = []
    for s in sentences:
        processed.extend(split_oversized_sentence(s, chunk_size))
    sentences = processed

    # Accumulate into chunks with overlap
    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        sentence_length = len(sentence)
        if current_length + sentence_length + 1 > chunk_size and current_chunk:
            chunks.append(" ".join(current_chunk))

            overlap_sentences = []
            overlap_length = 0
            for s in reversed(current_chunk):
                if overlap_length + len(s) > overlap:
                    break
                overlap_sentences.insert(0, s)
                overlap_length += len(s) + 1

            current_chunk = overlap_sentences
            current_length = overlap_length

        current_chunk.append(sentence)
        current_length += sentence_length + 1

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    # Quality filter
    if apply_quality_filter:
        before = len(chunks)
        chunks = [c for c in chunks if is_quality_chunk(c)]
        removed = before - len(chunks)
        if removed > 0:
            print(f"  Quality filter removed {removed} low-content chunk(s)")

    return chunks


# =============================================================================
# DOCUMENT CLASSIFIER AND BOILERPLATE SCORING (from Notebook 1, Cell 6)
# =============================================================================

def classify_document(text: str) -> str:
    """Classify document as 'academic' or 'general' from first ~2000 chars."""
    sample = text[:2000].lower()
    matches = sum(1 for signal in ACADEMIC_SIGNALS if signal in sample)
    doc_type = "academic" if matches >= ACADEMIC_SIGNAL_THRESHOLD else "general"
    print(f"  Classification: {doc_type} ({matches} academic signal(s))")
    return doc_type


def _keyword_score_academic(chunks: list[str]) -> list[float]:
    """Keyword-based boilerplate scores for academic documents."""
    scores = [0.0] * len(chunks)

    # Find content start
    content_start = 0
    for i, chunk in enumerate(chunks):
        chunk_lower = chunk.lower()
        if any(marker in chunk_lower for marker in ACADEMIC_CONTENT_MARKERS):
            content_start = i
            break
    for i in range(content_start):
        scores[i] = 1.0

    # Find references at the end
    reference_markers = ["references", "bibliography", "works cited"]
    content_end = len(chunks)
    scan_start = max(0, int(len(chunks) * 0.80))
    for i in range(len(chunks) - 1, scan_start - 1, -1):
        chunk_lower = chunks[i].lower()
        for marker in reference_markers:
            if marker in chunk_lower:
                content_end = i
                break
    for i in range(content_end, len(chunks)):
        scores[i] = 0.8

    return scores


def _keyword_score_general(chunks: list[str]) -> list[float]:
    """Keyword-based boilerplate scores for general documents."""
    scores = [0.0] * len(chunks)
    if chunks:
        first = chunks[0]
        letters = sum(1 for c in first if c.isalpha())
        ratio = letters / len(first) if len(first) > 0 else 0
        words = len(first.split())
        if words < 30 or ratio < 0.5:
            scores[0] = 0.7
    return scores


def _edge_repetition_scores(chunk_embeddings: np.ndarray, edge_count: int = 5) -> list[float]:
    """Score edge chunks by mutual similarity (formulaic language detection)."""
    n = len(chunk_embeddings)
    scores = [0.0] * n
    if n < edge_count * 2 + 4:
        return scores

    def pairwise_mean(embeds):
        sims = []
        for i in range(len(embeds)):
            for j in range(i + 1, len(embeds)):
                sims.append(cosine_similarity(embeds[i], embeds[j]))
        return np.mean(sims) if sims else 0.0

    def remap(val):
        return max(0.0, min(1.0, (val - 0.3) / 0.4))

    front_score = remap(pairwise_mean(chunk_embeddings[:edge_count]))
    back_score = remap(pairwise_mean(chunk_embeddings[-edge_count:]))

    for i in range(edge_count):
        scores[i] = front_score
    for i in range(n - edge_count, n):
        scores[i] = back_score

    return scores


def _divergence_scores(chunk_embeddings: np.ndarray, edge_count: int = 5) -> list[float]:
    """Score edge chunks by distance from the interior centroid."""
    n = len(chunk_embeddings)
    scores = [0.0] * n
    if n < edge_count * 2 + 4:
        return scores

    interior_start = int(n * 0.20)
    interior_end = int(n * 0.80)
    interior_centroid = np.mean(chunk_embeddings[interior_start:interior_end], axis=0)

    for i in range(edge_count):
        sim = cosine_similarity(chunk_embeddings[i], interior_centroid)
        scores[i] = max(0.0, min(1.0, (0.6 - sim) / 0.4))
    for i in range(n - edge_count, n):
        sim = cosine_similarity(chunk_embeddings[i], interior_centroid)
        scores[i] = max(0.0, min(1.0, (0.6 - sim) / 0.4))

    return scores


def score_boilerplate(
    chunks: list[str],
    doc_type: str,
    chunk_embeddings: np.ndarray,
    keyword_weight: float = 0.5,
    repetition_weight: float = 0.2,
    divergence_weight: float = 0.3,
) -> list[float]:
    """
    Combined boilerplate score per chunk (0.0–1.0).
    Three signals: keyword/positional, edge repetition, edge divergence.
    """
    if doc_type == "academic":
        kw_scores = _keyword_score_academic(chunks)
    else:
        kw_scores = _keyword_score_general(chunks)

    rep_scores = _edge_repetition_scores(chunk_embeddings)
    div_scores = _divergence_scores(chunk_embeddings)

    combined = []
    for i in range(len(chunks)):
        score = (
            keyword_weight * kw_scores[i] +
            repetition_weight * rep_scores[i] +
            divergence_weight * div_scores[i]
        )
        combined.append(max(0.0, min(1.0, score)))

    return combined


# =============================================================================
# CHROMADB INGESTION (from Notebook 1, Cell 7)
# =============================================================================

def ingest_document(
    chunks: list[str],
    chunk_embeddings: np.ndarray,
    boilerplate_scores: list[float],
    doc_id: str,
    doc_type: str,
    filename: str,
    collection_name: str = CHROMA_COLLECTION
) -> chromadb.Collection:
    """
    Store chunks + embeddings + metadata in ChromaDB.
    Clears existing collection on re-run (safe for iteration).
    """
    existing = chroma_client.get_or_create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"}
    )

    if existing.count() > 0:
        print(f"  ⚠ Collection '{collection_name}' has {existing.count()} entries — clearing")
        chroma_client.delete_collection(name=collection_name)
        existing = chroma_client.create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"}
        )

    ids = [f"{doc_id}_chunk_{i}" for i in range(len(chunks))]
    metadatas = [
        {
            "source": filename,
            "doc_id": doc_id,
            "doc_type": doc_type,
            "chunk_index": i,
            "char_count": len(chunk),
            "boilerplate_score": round(boilerplate_scores[i], 4)
        }
        for i, chunk in enumerate(chunks)
    ]

    BATCH_SIZE = 100
    for start in range(0, len(chunks), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(chunks))
        existing.add(
            ids=ids[start:end],
            embeddings=chunk_embeddings[start:end].tolist(),
            documents=chunks[start:end],
            metadatas=metadatas[start:end]
        )
        print(f"  Stored batch {start}–{end}")

    return existing


# =============================================================================
# TOP-LEVEL ENTRY POINT — the only new function in this cell
# =============================================================================

def upload_and_ingest(file_path: str) -> tuple[chromadb.Collection, dict]:
    """
    Full ingestion pipeline: load → chunk → classify → embed → score → store.

    This is the single entry point for getting a document into the system.
    Returns the ChromaDB collection and a metadata dict for reference.

    Args:
        file_path: Path to a PDF file

    Returns:
        (collection, doc_info) where doc_info contains:
            doc_id, filename, doc_type, num_chunks, num_pages
    """
    print(f"{'=' * 60}")
    print(f"INGESTION PIPELINE")
    print(f"{'=' * 60}")

    # Step 1: Load PDF
    doc = load_document(file_path)
    num_pages = len(PdfReader(file_path).pages)
    print(f"  Extracted {len(doc['text']):,} chars from {num_pages} pages\n")

    # Step 2: Chunk
    print("  Chunking...")
    chunks = chunk_text(doc["text"])
    print(f"  → {len(chunks)} chunks "
          f"(range: {min(len(c) for c in chunks)}–{max(len(c) for c in chunks)} chars)\n")

    # Step 3: Classify
    doc_type = classify_document(doc["text"])

    # Step 4: Embed
    print(f"\n  Embedding {len(chunks)} chunks...")
    chunk_embs = embedding_model.encode(chunks, show_progress_bar=True)
    print(f"  → Embeddings shape: {chunk_embs.shape}\n")

    # Step 5: Boilerplate scoring
    print("  Scoring boilerplate...")
    bp_scores = score_boilerplate(chunks, doc_type, chunk_embs)
    high_bp = sum(1 for s in bp_scores if s > 0.3)
    print(f"  → {high_bp} chunk(s) flagged (score > 0.3)\n")

    # Step 6: Ingest into ChromaDB
    print("  Ingesting into ChromaDB...")
    coll = ingest_document(
        chunks=chunks,
        chunk_embeddings=chunk_embs,
        boilerplate_scores=bp_scores,
        doc_id=doc["doc_id"],
        doc_type=doc_type,
        filename=doc["filename"]
    )

    # Summary
    doc_info = {
        "doc_id": doc["doc_id"],
        "filename": doc["filename"],
        "doc_type": doc_type,
        "num_chunks": len(chunks),
        "num_pages": num_pages
    }

    print(f"\n{'=' * 60}")
    print(f"✓ INGESTION COMPLETE")
    print(f"{'=' * 60}")
    print(f"  Doc ID:     {doc_info['doc_id']}")
    print(f"  File:       {doc_info['filename']}")
    print(f"  Type:       {doc_info['doc_type']}")
    print(f"  Pages:      {doc_info['num_pages']}")
    print(f"  Chunks:     {doc_info['num_chunks']}")
    print(f"  Collection: {CHROMA_COLLECTION} ({coll.count()} entries)")

    return coll, doc_info


# =============================================================================
# RUN: Ingest the sample document
# =============================================================================
SAMPLE_DOC_PATH = "./sample_doc.pdf"

collection, doc_info = upload_and_ingest(SAMPLE_DOC_PATH)

INGESTION PIPELINE
  Loading PDF: sample_doc.pdf
  Extracted 60,719 chars from 46 pages

  Chunking...
  Quality filter removed 6 low-content chunk(s)
  → 82 chunks (range: 301–799 chars)

  Classification: academic (6 academic signal(s))

  Embedding 82 chunks...


Batches: 100%|██████████| 3/3 [00:00<00:00, 14.57it/s]


  → Embeddings shape: (82, 384)

  Scoring boilerplate...
  → 5 chunk(s) flagged (score > 0.3)

  Ingesting into ChromaDB...
  ⚠ Collection 'rag_documents' has 82 entries — clearing
  Stored batch 0–82

✓ INGESTION COMPLETE
  Doc ID:     cab4fc8f6e6c
  File:       sample_doc.pdf
  Type:       academic
  Pages:      46
  Chunks:     82
  Collection: rag_documents (82 entries)


In [3]:
# =============================================================================
# CELL 3: Multi-User Session Manager
# =============================================================================
# Extends Notebook 1's SessionManager to support multiple users in one session.
#
# Key changes from Notebook 1:
#   - Messages now carry a "user_id" field alongside "role" and "content"
#   - role is either "user" or "assistant" (the bot)
#   - Bot messages use a reserved user_id: "agent"
#   - Added get_recent_messages() — returns last N messages regardless of role,
#     used by trigger detection (Cell 4) and throttling (Cell 7)
#   - Added get_bot_message_count() — counts bot messages in a window,
#     used by response throttling (Cell 7)
#   - Added messages_since_last_bot_response() — counts user messages since
#     the bot last spoke, used by throttling's minimum gap check
#
# Still in-memory — swap to Redis/SQLite for the FastAPI version.

AGENT_USER_ID = "agent"


class MultiUserSessionManager:
    """
    Manages multi-user conversation sessions.

    Each session stores a flat list of messages in chronological order:
        [
            {"role": "user",      "user_id": "alice", "content": "...", "timestamp": float},
            {"role": "user",      "user_id": "bob",   "content": "...", "timestamp": float},
            {"role": "assistant", "user_id": "agent",  "content": "...", "timestamp": float},
            ...
        ]

    "role" follows Ollama's convention (user/assistant) so history can be
    passed directly to /api/chat. "user_id" distinguishes who said what.
    """

    def __init__(self, max_history_turns: int = MAX_HISTORY_TURNS):
        self.sessions: dict[str, list[dict]] = {}
        self.max_history_turns = max_history_turns

    def create_session(self) -> str:
        """Create a new session and return its ID."""
        session_id = uuid.uuid4().hex[:12]
        self.sessions[session_id] = []
        print(f"  Session created: {session_id}")
        return session_id

    def add_user_message(self, session_id: str, user_id: str, content: str):
        """Record a message from a human user."""
        if session_id not in self.sessions:
            raise KeyError(f"Session '{session_id}' not found")

        self.sessions[session_id].append({
            "role": "user",
            "user_id": user_id,
            "content": content,
            "timestamp": time.time()
        })

    def add_bot_response(self, session_id: str, content: str):
        """Record a response from the agent."""
        if session_id not in self.sessions:
            raise KeyError(f"Session '{session_id}' not found")

        self.sessions[session_id].append({
            "role": "assistant",
            "user_id": AGENT_USER_ID,
            "content": content,
            "timestamp": time.time()
        })

    def get_recent_messages(self, session_id: str, n: int) -> list[dict]:
        """
        Get the last n messages (all roles) in chronological order.
        Used by trigger detection to see recent conversation context.
        """
        if session_id not in self.sessions:
            return []
        return self.sessions[session_id][-n:]

    def get_history_for_llm(self, session_id: str) -> list[dict]:
        """
        Get conversation history formatted for Ollama's /api/chat.
        Returns the last max_history_turns * 2 messages as
        [{"role": ..., "content": ...}] — strips user_id and timestamp
        since Ollama doesn't use them.

        Each message's content is prefixed with the user's name so the
        LLM knows who said what in the discussion.
        """
        if session_id not in self.sessions:
            return []

        messages = self.sessions[session_id]
        # Take last N messages (max_history_turns * 2 to roughly match
        # the old "turns" logic, but now turns aren't strictly paired)
        recent = messages[-(self.max_history_turns * 2):]

        formatted = []
        for msg in recent:
            if msg["role"] == "assistant":
                # Bot messages go as-is
                formatted.append({
                    "role": "assistant",
                    "content": msg["content"]
                })
            else:
                # User messages get prefixed with the speaker's name
                formatted.append({
                    "role": "user",
                    "content": f"[{msg['user_id']}]: {msg['content']}"
                })

        return formatted

    def messages_since_last_bot_response(self, session_id: str) -> int:
        """
        Count how many user messages have been sent since the bot last spoke.
        Returns a large number if the bot has never spoken.
        Used by response throttling (Cell 7).
        """
        if session_id not in self.sessions:
            return 999

        count = 0
        for msg in reversed(self.sessions[session_id]):
            if msg["role"] == "assistant":
                break
            count += 1

        return count

    def get_bot_message_ratio(self, session_id: str, window: int = RATIO_WINDOW) -> float:
        """
        Fraction of the last `window` messages that are bot responses.
        Used by response throttling to detect if the bot is dominating.
        """
        if session_id not in self.sessions:
            return 0.0

        recent = self.sessions[session_id][-window:]
        if not recent:
            return 0.0

        bot_count = sum(1 for m in recent if m["role"] == "assistant")
        return bot_count / len(recent)

    def get_full_log(self, session_id: str) -> list[dict]:
        """Return the complete message log for a session."""
        return self.sessions.get(session_id, [])

    def list_sessions(self) -> dict:
        """Return all sessions with message counts."""
        return {sid: len(msgs) for sid, msgs in self.sessions.items()}

    def delete_session(self, session_id: str):
        """Delete a session entirely."""
        self.sessions.pop(session_id, None)


# =============================================================================
# Initialize the global session manager
# =============================================================================
session_manager = MultiUserSessionManager()

# =============================================================================
# TEST: Simulate a multi-user conversation
# =============================================================================
print(f"{'=' * 60}")
print("TEST: Multi-user session manager")
print(f"{'=' * 60}\n")

# Create session
test_session = session_manager.create_session()

# Simulate a discussion between two users and the bot
test_messages = [
    ("alice", "Hey, I just read through the paper. Interesting stuff."),
    ("bob",   "Yeah, what did they use for the emotion detection part?"),
    ("alice", "I think it was some kind of CNN but I'm not sure."),
    # Bot would interject here in the real system
    ("bob",   "Makes sense. What about the deployment?"),
    ("alice", "Docker I think? Or maybe Kubernetes."),
]

for user_id, content in test_messages:
    session_manager.add_user_message(test_session, user_id, content)

# Simulate a bot response after message 3
session_manager.add_bot_response(
    test_session,
    "Based on the paper, the system uses a CNN-LSTM hybrid model for emotion detection — "
    "see section 3.2 for the full architecture details."
)

# --- Verify message log ---
log = session_manager.get_full_log(test_session)
print(f"Total messages: {len(log)}\n")

for msg in log:
    role_tag = "BOT" if msg["role"] == "assistant" else msg["user_id"].upper()
    print(f"  [{role_tag}] {msg['content'][:80]}{'...' if len(msg['content']) > 80 else ''}")

# --- Verify LLM-formatted history ---
print(f"\n{'=' * 60}")
print("LLM-FORMATTED HISTORY (as Ollama will see it):")
print(f"{'=' * 60}")

llm_history = session_manager.get_history_for_llm(test_session)
for msg in llm_history:
    print(f"  {msg['role']:>10}: {msg['content'][:80]}{'...' if len(msg['content']) > 80 else ''}")

# --- Verify throttling helpers ---
print(f"\n{'=' * 60}")
print("THROTTLING HELPERS:")
print(f"{'=' * 60}")

gap = session_manager.messages_since_last_bot_response(test_session)
ratio = session_manager.get_bot_message_ratio(test_session)
print(f"  Messages since last bot response: {gap}")
print(f"  Bot message ratio (last {RATIO_WINDOW}): {ratio:.1%}")

# --- Verify get_recent_messages (used by trigger detection) ---
print(f"\n{'=' * 60}")
print(f"RECENT MESSAGES (last {TRIGGER_CONTEXT_WINDOW}):")
print(f"{'=' * 60}")

recent = session_manager.get_recent_messages(test_session, TRIGGER_CONTEXT_WINDOW)
for msg in recent:
    role_tag = "BOT" if msg["role"] == "assistant" else msg["user_id"]
    print(f"  [{role_tag}] {msg['content'][:80]}")

print(f"\n✓ Multi-user session manager working — ready for Cell 4 (trigger detection)")

TEST: Multi-user session manager

  Session created: ba281b60f240
Total messages: 6

  [ALICE] Hey, I just read through the paper. Interesting stuff.
  [BOB] Yeah, what did they use for the emotion detection part?
  [ALICE] I think it was some kind of CNN but I'm not sure.
  [BOB] Makes sense. What about the deployment?
  [ALICE] Docker I think? Or maybe Kubernetes.
  [BOT] Based on the paper, the system uses a CNN-LSTM hybrid model for emotion detectio...

LLM-FORMATTED HISTORY (as Ollama will see it):
        user: [alice]: Hey, I just read through the paper. Interesting stuff.
        user: [bob]: Yeah, what did they use for the emotion detection part?
        user: [alice]: I think it was some kind of CNN but I'm not sure.
        user: [bob]: Makes sense. What about the deployment?
        user: [alice]: Docker I think? Or maybe Kubernetes.
   assistant: Based on the paper, the system uses a CNN-LSTM hybrid model for emotion detectio...

THROTTLING HELPERS:
  Messages since last b

In [4]:
# =============================================================================
# CELL 4: Trigger Detection / Intent Classifier
# =============================================================================
# The core novelty of the discussion agent. For each new message, decides:
#   "Should the bot respond to this?"
#
# Two-stage approach:
#   Stage 1 — Rule-based fast path:
#       Catches direct @bot mentions and obvious non-triggers (very short
#       messages, pure agreement, casual chat) WITHOUT an LLM call.
#       This is instant and saves ~3-10 seconds per non-trigger message.
#
#   Stage 2 — LLM-based classification:
#       For ambiguous messages, sends the message + recent context to Ollama
#       with a short classification prompt. The LLM responds YES or NO with
#       a trigger type. This is cheaper than a full RAG call — no retrieval,
#       short prompt, short response.
#
# Trigger types:
#   DIRECT_TAG    — user explicitly addresses the bot (@bot, "hey bot", etc.)
#   QUESTION      — question about document content
#   UNCERTAINTY   — speculation or unsure statement that the document could resolve
#   DISAGREEMENT  — two users contradict each other on a factual point
#   CLARIFICATION — request for explanation the document could provide
#   NONE          — no trigger (bot stays silent)

# =============================================================================
# CONFIGURATION
# =============================================================================

# Bot name patterns for direct tag detection (case-insensitive)
BOT_NAMES = ["@bot", "hey bot", "bot,", "bot:"]

# Rule-based non-trigger patterns — messages matching these skip the LLM call
# These are checked as lowercase prefixes or full matches
NON_TRIGGER_PHRASES = [
    "yeah", "yep", "yes", "sure", "ok", "okay", "right",
    "agreed", "exactly", "makes sense", "good point", "true",
    "lol", "haha", "hah", "lmao",
    "let's take a break", "let's move on", "brb", "back",
    "should we move on", "next topic", "let's continue",
    "thanks", "thank you", "cool", "nice", "great",
]

# LLM classification prompt — kept minimal for speed
TRIGGER_CLASSIFIER_SYSTEM_PROMPT = """You are a conversation monitor. Your job is to decide whether a new message in a group discussion requires a factual response from a document-aware assistant.

The assistant has access to an uploaded document and should ONLY respond when document knowledge would be helpful.

Respond with EXACTLY one line in this format:
DECISION: YES or NO
TYPE: DIRECT_TAG | QUESTION | UNCERTAINTY | DISAGREEMENT | CLARIFICATION | NONE
REASON: one short phrase (max 5 words)

Triggers (respond YES):
- Direct mention of the bot/assistant
- Questions about document content (methods, results, tools, data, architecture)
- Uncertainty or speculation about something the document could clarify ("I think they used X but not sure")
- Factual disagreement between users that the document could resolve
- Requests for explanation of something covered in the document

Non-triggers (respond NO):
- Opinions and preferences ("I think this approach is better")
- Agreement or acknowledgment ("yeah", "makes sense", "good point")
- Casual chat, greetings, meta-discussion ("let's take a break", "should we move on")
- Questions unrelated to any uploaded document content"""


def _rule_based_check(message: str) -> tuple[bool, str] | None:
    """
    Fast-path rule-based trigger detection. No LLM call needed.

    Returns:
        ("trigger", trigger_type) if definitely a trigger
        ("skip", "NONE") if definitely NOT a trigger
        None if ambiguous — needs LLM classification
    """
    msg_lower = message.strip().lower()

    # --- Direct tag: always trigger ---
    for bot_name in BOT_NAMES:
        if bot_name in msg_lower:
            return ("trigger", "DIRECT_TAG")

    # --- Very short messages: almost never triggers ---
    # Single words or very short phrases are usually reactions
    if len(msg_lower.split()) <= 2:
        # Unless it's a short question
        if msg_lower.endswith("?"):
            return None  # Ambiguous — let LLM decide
        return ("skip", "NONE")

    # --- Known non-trigger phrases ---
    for phrase in NON_TRIGGER_PHRASES:
        if msg_lower == phrase or msg_lower.startswith(phrase + " "):
            return ("skip", "NONE")

    # --- Everything else is ambiguous ---
    return None


def _llm_classify(message: str, context_messages: list[dict]) -> dict:
    """
    Use the LLM to classify whether a message should trigger a bot response.

    Args:
        message:          The new message to classify
        context_messages: Recent conversation messages for context

    Returns:
        {"decision": "YES"|"NO", "type": str, "reason": str}
    """
    # Build context string from recent messages
    context_lines = []
    for msg in context_messages:
        if msg["role"] == "assistant":
            context_lines.append(f"[assistant]: {msg['content']}")
        else:
            context_lines.append(f"[{msg['user_id']}]: {msg['content']}")

    context_str = "\n".join(context_lines) if context_lines else "(no prior context)"

    user_prompt = f"""Recent conversation:
{context_str}

New message to classify:
{message}

Should the document-aware assistant respond to this new message?"""

    try:
        response = requests.post(
            f"{OLLAMA_BASE_URL}/api/chat",
            json={
                "model": OLLAMA_MODEL,
                "messages": [
                    {"role": "system", "content": TRIGGER_CLASSIFIER_SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                "stream": False
            },
            timeout=30
        )

        raw = response.json()["message"]["content"].strip()

        # Parse the structured response
        result = {"decision": "NO", "type": "NONE", "reason": "parse_failed"}

        # Valid trigger types in priority order — when the LLM returns
        # multiple types (e.g. "DIRECT_TAG | QUESTION"), we pick the
        # most specific one. Ordered from most to least specific.
        valid_types_priority = [
            "DISAGREEMENT", "UNCERTAINTY", "CLARIFICATION",
            "QUESTION", "DIRECT_TAG", "NONE"
        ]

        for line in raw.split("\n"):
            line = line.strip()
            if line.upper().startswith("DECISION:"):
                val = line.split(":", 1)[1].strip().upper()
                result["decision"] = "YES" if "YES" in val else "NO"
            elif line.upper().startswith("TYPE:"):
                type_str = line.split(":", 1)[1].strip().upper()
                # Pick the most specific valid type found in the response
                matched = [t for t in valid_types_priority if t in type_str]
                result["type"] = matched[0] if matched else "NONE"
            elif line.upper().startswith("REASON:"):
                result["reason"] = line.split(":", 1)[1].strip()

        # Enforce consistency: if decision is NO, type must be NONE
        if result["decision"] == "NO":
            result["type"] = "NONE"

        # Override: if LLM says DIRECT_TAG but the message doesn't
        # actually mention the bot, downgrade to QUESTION
        if result["type"] == "DIRECT_TAG":
            msg_lower = message.strip().lower()
            if not any(name in msg_lower for name in BOT_NAMES):
                result["type"] = "QUESTION"

        return result

    except Exception as e:
        print(f"  ⚠ Trigger classifier error: {e}")
        # Fail safe — don't trigger on errors
        return {"decision": "NO", "type": "NONE", "reason": f"error: {e}"}


def detect_trigger(
    session_id: str,
    message: str,
    user_id: str
) -> dict:
    """
    Determine whether a new message should trigger a bot response.

    Two-stage pipeline:
        1. Rule-based fast path (instant)
        2. LLM classification (3-10 seconds, only if needed)

    Args:
        session_id: The active session
        message:    The new message text
        user_id:    Who sent it

    Returns:
        {
            "should_respond": bool,
            "trigger_type":   str (DIRECT_TAG|QUESTION|UNCERTAINTY|DISAGREEMENT|CLARIFICATION|NONE),
            "reason":         str,
            "method":         "rule" or "llm"
        }
    """
    # Stage 1: Rule-based fast path
    rule_result = _rule_based_check(message)

    if rule_result is not None:
        action, trigger_type = rule_result
        return {
            "should_respond": action == "trigger",
            "trigger_type": trigger_type,
            "reason": "direct_mention" if trigger_type == "DIRECT_TAG" else "rule_filtered",
            "method": "rule"
        }

    # Stage 2: LLM classification
    # Get recent messages for context (excluding the current message,
    # which hasn't been added to the session yet)
    context = session_manager.get_recent_messages(session_id, TRIGGER_CONTEXT_WINDOW)

    llm_result = _llm_classify(message, context)

    return {
        "should_respond": llm_result["decision"] == "YES",
        "trigger_type": llm_result["type"],
        "reason": llm_result["reason"],
        "method": "llm"
    }


# =============================================================================
# TEST: Verify trigger detection on sample messages
# =============================================================================
print(f"{'=' * 60}")
print("TEST: Trigger Detection")
print(f"{'=' * 60}\n")

# Create a test session with some conversation context
test_session_trigger = session_manager.create_session()

# Add some context so the LLM classifier has something to work with
context_setup = [
    ("alice", "I've been reading through the paper on the emotion detection system."),
    ("bob",   "Same, the architecture section was interesting."),
]
for uid, content in context_setup:
    session_manager.add_user_message(test_session_trigger, uid, content)

# Test cases: (user_id, message, expected_trigger)
test_cases = [
    # --- Should trigger ---
    ("alice", "@bot what model did they use?",                          True,  "DIRECT_TAG (rule)"),
    ("bob",   "What dataset was used for training?",                    True,  "QUESTION (llm)"),
    ("alice", "I think they used Docker but I'm not sure",              True,  "UNCERTAINTY (llm)"),
    ("bob",   "Can someone explain the deployment architecture?",       True,  "CLARIFICATION (llm)"),

    # --- Should NOT trigger ---
    ("alice", "Yeah that makes sense",                                  False, "agreement (rule)"),
    ("bob",   "Let's take a break",                                     False, "casual (rule)"),
    ("alice", "I think the CNN approach is better than transformers",    False, "opinion (llm)"),
    ("bob",   "ok",                                                     False, "short msg (rule)"),
]

print(f"Running {len(test_cases)} test cases...\n")

results = []
for user_id, message, expected, description in test_cases:
    start = time.time()
    result = detect_trigger(test_session_trigger, message, user_id)
    elapsed = time.time() - start

    match = result["should_respond"] == expected
    status = "✓" if match else "✗"

    results.append(match)

    print(f"  {status} [{result['method']:4s}] {elapsed:5.2f}s | "
          f"trigger={str(result['should_respond']):5s} | "
          f"type={result['trigger_type']:<15s} | "
          f"\"{message[:55]}{'...' if len(message) > 55 else ''}\"")
    if not match:
        print(f"    ↳ EXPECTED trigger={expected} ({description})")

# Summary
passed = sum(results)
total = len(results)
rule_cases = sum(1 for _, m, _, _ in test_cases for r in [_rule_based_check(m)] if r is not None)

print(f"\n{'=' * 60}")
print(f"RESULTS: {passed}/{total} correct")
print(f"  Rule-based (instant): {rule_cases}/{total} messages handled without LLM")
print(f"  LLM-classified:       {total - rule_cases}/{total} messages needed LLM call")
print(f"{'=' * 60}")

if passed == total:
    print(f"\n✓ All trigger tests passed — ready for Cell 5 (query reformulation)")
else:
    print(f"\n⚠ {total - passed} test(s) failed — review LLM classifications above")
    print(f"  Note: LLM classification is non-deterministic. Minor mismatches on")
    print(f"  borderline cases (opinions, vague questions) are expected. The rule-based")
    print(f"  fast path handles the clear-cut cases deterministically.")

TEST: Trigger Detection

  Session created: f8048d080da8
Running 8 test cases...

  ✓ [rule]  0.00s | trigger=True  | type=DIRECT_TAG      | "@bot what model did they use?"
  ✓ [llm ]  5.89s | trigger=True  | type=QUESTION        | "What dataset was used for training?"
  ✓ [llm ]  2.57s | trigger=True  | type=UNCERTAINTY     | "I think they used Docker but I'm not sure"
  ✓ [llm ]  2.66s | trigger=True  | type=CLARIFICATION   | "Can someone explain the deployment architecture?"
  ✓ [rule]  0.00s | trigger=False | type=NONE            | "Yeah that makes sense"
  ✓ [rule]  0.00s | trigger=False | type=NONE            | "Let's take a break"
  ✓ [llm ]  2.65s | trigger=False | type=NONE            | "I think the CNN approach is better than transformers"
  ✓ [rule]  0.00s | trigger=False | type=NONE            | "ok"

RESULTS: 8/8 correct
  Rule-based (instant): 4/8 messages handled without LLM
  LLM-classified:       4/8 messages needed LLM call

✓ All trigger tests passed — ready for Cell

In [5]:
# =============================================================================
# CELL 5: Discussion-Aware Query Reformulation
# =============================================================================
# Problem: In a group discussion, messages are often vague or contextual.
#   "What about the accuracy?" only makes sense if you know the last few
#   messages were about emotion detection models. Retrieving chunks for
#   just "accuracy" gives garbage results.
#
# Solution: Before retrieval, use the conversation context to reformulate
#   the trigger message into a self-contained, retrieval-ready query.
#
# Two-stage approach (same pattern as trigger detection):
#   Stage 1 — Skip reformulation if the message is already specific enough.
#       Direct @bot questions with enough detail, or messages over a certain
#       word count with clear nouns/terms, go straight to retrieval as-is.
#   Stage 2 — LLM reformulation for vague/contextual messages.
#       Send the trigger message + recent conversation to Ollama and ask
#       it to produce a single, self-contained search query.
#
# This sits between trigger detection (Cell 4) and the RAG pipeline (Cell 6).
# Flow: message → trigger? → reformulate → retrieve → generate

# =============================================================================
# CONFIGURATION
# =============================================================================

REFORMULATION_SYSTEM_PROMPT = """You are a search query optimizer. Your job is to rewrite a conversational message into a clear, self-contained search query for retrieving information from an academic/technical document.

You will see recent conversation context and a new message. Rewrite the new message so it:
1. Replaces pronouns (it, they, that, this) with the actual subjects from context
2. Includes key topic terms from the discussion
3. Is a single, concise query (under 20 words)
4. Works as a standalone search — someone reading ONLY the query should understand what's being asked

Respond with ONLY the reformulated query. No explanation, no quotes, no preamble."""

# Minimum word count to consider a message "specific enough" to skip reformulation.
# Messages with clear technical terms and enough words don't need rewriting.
MIN_SPECIFIC_WORDS = 8

# Words that signal vagueness — if a short message contains these,
# it almost certainly needs reformulation
# Multi-word phrases checked with substring matching (safe, no false positives)
VAGUE_PHRASE_SIGNALS = [
    "what about", "how about", "and the", "the part about",
    "the thing about", "the bit about"
]

# Single-word pronouns checked with word-boundary matching to avoid
# false positives like "it" matching "arch-it-ecture" or "them" in "theme"
VAGUE_WORD_SIGNALS = ["it", "that", "this", "they", "them", "those"]


def _needs_reformulation(message: str) -> bool:
    """
    Decide whether a message needs LLM reformulation or is already
    specific enough for direct retrieval.

    Returns True if reformulation is needed, False if the message
    can go straight to retrieval.
    """
    msg_lower = message.strip().lower()
    words = msg_lower.split()

    # Short messages almost always need context
    if len(words) < MIN_SPECIFIC_WORDS:
        return True

    # Check for vague multi-word phrases (substring match is safe here)
    for phrase in VAGUE_PHRASE_SIGNALS:
        if phrase in msg_lower:
            return True

    # Check for vague pronouns using word boundaries to avoid
    # matching inside longer words (e.g. "it" in "architecture")
    for pronoun in VAGUE_WORD_SIGNALS:
        if re.search(rf'\b{pronoun}\b', msg_lower):
            return True

    # Long enough and no vague signals — specific enough for direct retrieval
    return False


def reformulate_query(
    session_id: str,
    message: str,
    user_id: str
) -> dict:
    """
    Reformulate a trigger message into a retrieval-ready query.

    If the message is already specific, returns it unchanged.
    Otherwise, uses the LLM to expand it with conversation context.

    Args:
        session_id: Active session for conversation context
        message:    The trigger message to reformulate
        user_id:    Who sent it

    Returns:
        {
            "original":      str,  the trigger message
            "reformulated":  str,  the retrieval-ready query
            "was_rewritten": bool, whether the LLM was used
        }
    """
    # Stage 1: Check if reformulation is needed
    if not _needs_reformulation(message):
        return {
            "original": message,
            "reformulated": message,
            "was_rewritten": False
        }

    # Stage 2: LLM reformulation
    # Get recent conversation context
    recent = session_manager.get_recent_messages(
        session_id, REFORMULATION_CONTEXT_TURNS * 2
    )

    # Build context string
    context_lines = []
    for msg in recent:
        if msg["role"] == "assistant":
            context_lines.append(f"[assistant]: {msg['content']}")
        else:
            context_lines.append(f"[{msg['user_id']}]: {msg['content']}")

    context_str = "\n".join(context_lines) if context_lines else "(no prior context)"

    user_prompt = f"""Recent conversation:
{context_str}

New message to reformulate:
[{user_id}]: {message}

Rewrite this as a clear, self-contained search query:"""

    try:
        response = requests.post(
            f"{OLLAMA_BASE_URL}/api/chat",
            json={
                "model": OLLAMA_MODEL,
                "messages": [
                    {"role": "system", "content": REFORMULATION_SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                "stream": False
            },
            timeout=30
        )

        reformulated = response.json()["message"]["content"].strip()

        # Clean up: remove quotes the LLM might wrap around the query
        reformulated = reformulated.strip('"').strip("'").strip()

        # Safety: if the LLM returns something absurdly long or empty,
        # fall back to the original message
        if len(reformulated) == 0 or len(reformulated.split()) > 30:
            return {
                "original": message,
                "reformulated": message,
                "was_rewritten": False
            }

        return {
            "original": message,
            "reformulated": reformulated,
            "was_rewritten": True
        }

    except Exception as e:
        print(f"  ⚠ Reformulation error: {e}")
        # Fail safe — use original message
        return {
            "original": message,
            "reformulated": message,
            "was_rewritten": False
        }


# =============================================================================
# TEST: Verify query reformulation
# =============================================================================
print(f"{'=' * 60}")
print("TEST: Discussion-Aware Query Reformulation")
print(f"{'=' * 60}\n")

# Create a session with realistic discussion context
test_session_reform = session_manager.create_session()

# Simulate a conversation about emotion detection
context_messages = [
    ("alice", "The paper describes an emotion detection system using deep learning."),
    ("bob",   "Right, and they tested it on multiple datasets for facial expressions."),
    ("alice", "The CNN part was interesting — they combined it with an LSTM."),
    ("bob",   "I wonder how well it actually performed."),
]
for uid, content in context_messages:
    session_manager.add_user_message(test_session_reform, uid, content)

# Test cases: (user_id, message, description)
test_cases = [
    # --- Should be reformulated (vague/contextual) ---
    ("alice", "What about the accuracy?",
     "Vague — 'the accuracy' needs context"),
    ("bob",   "How did they train it?",
     "Short + pronoun 'it' needs resolution"),
    ("alice", "What datasets were used for that?",
     "Pronoun 'that' needs context"),
    ("bob",   "And the deployment?",
     "Fragment — needs full topic from context"),

    # --- Should NOT be reformulated (already specific) ---
    ("alice", "What CNN architecture was used for the emotion detection model?",
     "Specific enough — clear subject and topic"),
    ("bob",   "What accuracy did the facial expression recognition system achieve on the FER2013 dataset?",
     "Very specific — full context in the message itself"),
]

print(f"Conversation context:")
for uid, content in context_messages:
    print(f"  [{uid}] {content}")
print()

for user_id, message, description in test_cases:
    start = time.time()
    result = reformulate_query(test_session_reform, message, user_id)
    elapsed = time.time() - start

    rewrite_tag = "REWRITTEN" if result["was_rewritten"] else "AS-IS"

    print(f"  [{rewrite_tag:9s}] {elapsed:5.2f}s")
    print(f"    Original:     \"{message}\"")
    if result["was_rewritten"]:
        print(f"    Reformulated: \"{result['reformulated']}\"")
    print(f"    ({description})")
    print()

print(f"✓ Query reformulation complete — ready for Cell 6 (dual-context RAG pipeline)")

TEST: Discussion-Aware Query Reformulation

  Session created: 3a8d9c7f1a66
Conversation context:
  [alice] The paper describes an emotion detection system using deep learning.
  [bob] Right, and they tested it on multiple datasets for facial expressions.
  [alice] The CNN part was interesting — they combined it with an LSTM.
  [bob] I wonder how well it actually performed.

  [REWRITTEN]  2.70s
    Original:     "What about the accuracy?"
    Reformulated: "Emotion detection system performance accuracy deep learning facial expressions CNN LSTM"
    (Vague — 'the accuracy' needs context)

  [REWRITTEN]  2.58s
    Original:     "How did they train it?"
    Reformulated: "Train the deep learning emotion detection system using CNN-LSTM combination on multiple datasets."
    (Short + pronoun 'it' needs resolution)

  [REWRITTEN]  2.46s
    Original:     "What datasets were used for that?"
    Reformulated: "Emotion detection system datasets for facial expressions testing"
    (Pronoun 'tha

In [6]:
# =============================================================================
# CELL 6: Dual-Context RAG Pipeline
# =============================================================================
# The agent's response generator. Combines two context sources:
#   1. Document context — retrieved chunks relevant to the trigger message
#   2. Conversation context — recent discussion history
#
# This replaces Notebook 1's rag_query() and rag_query_with_history() with
# a single function designed for the discussion agent's collaborative tone.
#
# Pipeline flow (called only when trigger detection says YES):
#   trigger message → reformulate (Cell 5) → retrieve → build dual prompt → generate
#
# Key differences from Notebook 1's rag_query_with_history():
#   - Uses reformulated query for retrieval (not the raw message)
#   - System prompt has a collaborative, conversational tone
#   - Conversation history includes multi-user speaker labels
#   - Returns metadata for throttling and logging

# =============================================================================
# RETRIEVAL (carried over from Notebook 1 Cell 8)
# =============================================================================

def retrieve(query: str, top_k: int = TOP_K, boilerplate_penalty: float = 0.5) -> dict:
    """
    Retrieve the most relevant chunks for a query, with boilerplate demotion.
    Fetches top_k * 2 candidates, penalizes by boilerplate_score, returns best top_k.
    """
    prefixed_query = BGE_QUERY_PREFIX + query
    query_embedding = embedding_model.encode(prefixed_query).tolist()

    candidate_count = min(top_k * 2, collection.count())
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=candidate_count,
        include=["documents", "metadatas", "distances"]
    )

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]
    ids = results["ids"][0]

    adjusted_scores = []
    for dist, meta in zip(distances, metadatas):
        base_score = 1 - dist
        bp_score = meta.get("boilerplate_score", 0.0)
        adjusted = base_score * (1 - boilerplate_penalty * bp_score)
        adjusted_scores.append(adjusted)

    ranked = sorted(
        zip(documents, metadatas, distances, adjusted_scores, ids),
        key=lambda x: x[3],
        reverse=True
    )[:top_k]

    return {
        "documents":       [r[0] for r in ranked],
        "metadatas":       [r[1] for r in ranked],
        "distances":       [r[2] for r in ranked],
        "adjusted_scores": [r[3] for r in ranked],
        "ids":             [r[4] for r in ranked]
    }


def format_context(retrieved: dict) -> str:
    """Format retrieved chunks into a context string for the LLM prompt."""
    context_parts = []
    for i, (doc_text, meta, adj_score) in enumerate(zip(
        retrieved["documents"],
        retrieved["metadatas"],
        retrieved["adjusted_scores"]
    )):
        bp = meta.get("boilerplate_score", 0.0)
        bp_tag = " [low-confidence source]" if bp > 0.3 else ""

        context_parts.append(
            f"[Source: {meta['source']}, Chunk {meta['chunk_index']}, "
            f"Relevance: {adj_score:.2%}{bp_tag}]\n{doc_text}"
        )

    return "\n\n---\n\n".join(context_parts)


# =============================================================================
# DISCUSSION AGENT SYSTEM PROMPT
# =============================================================================
# Collaborative tone — the bot is a knowledgeable colleague, not a search engine.
# Key differences from Notebook 1's SYSTEM_PROMPT:
#   - Speaks as a participant in a group discussion
#   - Encourages natural phrasing ("Based on the paper, ..." not "According to Chunk 12, ...")
#   - Still grounded — won't make things up
#   - Aware of low-confidence sources

DISCUSSION_SYSTEM_PROMPT = """You are a knowledgeable assistant participating in a team discussion. You have access to an uploaded document and are helping the team understand its contents.

GUIDELINES:
1. Be conversational — you're a colleague contributing to the discussion, not a search engine.
2. Ground your responses in the provided document context. Reference sections or findings naturally (e.g. "Based on the paper, the system uses..." or "Section 4 describes...").
3. If the context doesn't contain enough information, say so honestly — don't guess or make things up.
4. Keep responses focused and concise. The team is having a discussion, not reading an essay.
5. If a source is tagged [low-confidence source], treat it with caution and prefer other sources.
6. When correcting a misconception, be tactful — e.g. "Actually, the paper mentions..." rather than "You're wrong."
7. Address the specific point being discussed. Don't dump everything you know about a topic."""


def discussion_rag_query(
    session_id: str,
    message: str,
    user_id: str,
    trigger_type: str,
    top_k: int = TOP_K
) -> dict:
    """
    Full discussion-aware RAG pipeline: reformulate → retrieve → generate.

    Called only when trigger detection (Cell 4) returns should_respond=True.
    Uses the reformulated query for retrieval but passes the original message
    to the LLM so it can respond naturally to what the user actually said.

    Args:
        session_id:   Active session
        message:      The original trigger message (what the user said)
        user_id:      Who sent it
        trigger_type: From Cell 4 (QUESTION, UNCERTAINTY, etc.) — for logging
        top_k:        Number of chunks to retrieve

    Returns:
        {
            "answer":           str,   the bot's response
            "original_message": str,   what the user said
            "reformulated":     str,   query used for retrieval
            "was_rewritten":    bool,  whether reformulation changed the query
            "trigger_type":     str,   from Cell 4
            "sources":          list,  chunk metadata
            "scores":           list,  adjusted relevance scores
            "session_id":       str
        }
    """
    # Step 1: Reformulate the message for better retrieval
    reform = reformulate_query(session_id, message, user_id)

    # Step 2: Retrieve relevant chunks using the reformulated query
    retrieved = retrieve(reform["reformulated"], top_k=top_k)
    context = format_context(retrieved)

    # Step 3: Build the dual-context prompt
    # The user message includes document context + the original message
    # (not the reformulated one — we want the LLM to respond to what
    # the user actually said, not the search query version of it)
    user_prompt = f"""Document context:
{context}

---

The following is a team discussion. Respond to the latest message using the document context above.

Latest message from {user_id}: {message}"""

    # Step 4: Assemble messages array with conversation history
    messages = [{"role": "system", "content": DISCUSSION_SYSTEM_PROMPT}]

    # Add conversation history (multi-user formatted)
    history = session_manager.get_history_for_llm(session_id)
    messages.extend(history)

    # Add the current context-augmented message
    messages.append({"role": "user", "content": user_prompt})

    # Step 5: Generate response
    response = requests.post(
        f"{OLLAMA_BASE_URL}/api/chat",
        json={
            "model": OLLAMA_MODEL,
            "messages": messages,
            "stream": False
        },
        timeout=60
    )

    data = response.json()
    answer = data["message"]["content"]

    # Step 6: Store the bot's response in session history
    # The user's message was already added to the session by the orchestrator
    # (Cell 8), so we only add the bot's response here.
    session_manager.add_bot_response(session_id, answer)

    return {
        "answer": answer,
        "original_message": message,
        "reformulated": reform["reformulated"],
        "was_rewritten": reform["was_rewritten"],
        "trigger_type": trigger_type,
        "sources": retrieved["metadatas"],
        "scores": retrieved["adjusted_scores"],
        "session_id": session_id
    }


# =============================================================================
# TEST: Verify the dual-context pipeline
# =============================================================================
print(f"{'=' * 60}")
print("TEST: Dual-Context RAG Pipeline")
print(f"{'=' * 60}\n")

# Create a session with discussion context
test_session_rag = session_manager.create_session()

# Build up a realistic conversation
setup_messages = [
    ("alice", "I've been reading the paper on the emotion detection system."),
    ("bob",   "The deep learning architecture looked complex."),
    ("alice", "Yeah, they combined multiple model types together."),
]
for uid, content in setup_messages:
    session_manager.add_user_message(test_session_rag, uid, content)

print("Conversation context:")
for uid, content in setup_messages:
    print(f"  [{uid}] {content}")

# --- Test 1: Vague message that needs reformulation ---
print(f"\n{'=' * 60}")
print("TEST 1: Vague message (needs reformulation)")
print(f"{'=' * 60}")

test_msg_1 = "What about the accuracy?"
test_user_1 = "bob"

# Add the user's message to the session first (simulating the orchestrator)
session_manager.add_user_message(test_session_rag, test_user_1, test_msg_1)

start = time.time()
result_1 = discussion_rag_query(
    test_session_rag, test_msg_1, test_user_1, "QUESTION"
)
elapsed_1 = time.time() - start

print(f"\n  [{test_user_1}]: {test_msg_1}")
print(f"  Reformulated: \"{result_1['reformulated']}\"")
print(f"  Rewritten:    {result_1['was_rewritten']}")
print(f"\n  [bot]: {result_1['answer'][:400]}{'...' if len(result_1['answer']) > 400 else ''}")
print(f"\n  Sources: {[m['chunk_index'] for m in result_1['sources']]}")
print(f"  Scores:  {[f'{s:.2%}' for s in result_1['scores']]}")
print(f"  Time:    {elapsed_1:.2f}s")

# --- Test 2: Specific message (no reformulation needed) ---
print(f"\n{'=' * 60}")
print("TEST 2: Specific message (no reformulation)")
print(f"{'=' * 60}")

test_msg_2 = "@bot What deep learning model architecture is used for emotion detection?"
test_user_2 = "alice"

session_manager.add_user_message(test_session_rag, test_user_2, test_msg_2)

start = time.time()
result_2 = discussion_rag_query(
    test_session_rag, test_msg_2, test_user_2, "DIRECT_TAG"
)
elapsed_2 = time.time() - start

print(f"\n  [{test_user_2}]: {test_msg_2}")
print(f"  Rewritten:    {result_2['was_rewritten']}")
print(f"\n  [bot]: {result_2['answer'][:400]}{'...' if len(result_2['answer']) > 400 else ''}")
print(f"\n  Sources: {[m['chunk_index'] for m in result_2['sources']]}")
print(f"  Scores:  {[f'{s:.2%}' for s in result_2['scores']]}")
print(f"  Time:    {elapsed_2:.2f}s")

# --- Test 3: Uncertainty that the document can resolve ---
print(f"\n{'=' * 60}")
print("TEST 3: Uncertainty (document can resolve)")
print(f"{'=' * 60}")

test_msg_3 = "I think they used Docker for deployment but I'm not sure"
test_user_3 = "bob"

session_manager.add_user_message(test_session_rag, test_user_3, test_msg_3)

start = time.time()
result_3 = discussion_rag_query(
    test_session_rag, test_msg_3, test_user_3, "UNCERTAINTY"
)
elapsed_3 = time.time() - start

print(f"\n  [{test_user_3}]: {test_msg_3}")
if result_3['was_rewritten']:
    print(f"  Reformulated: \"{result_3['reformulated']}\"")
print(f"  Rewritten:    {result_3['was_rewritten']}")
print(f"\n  [bot]: {result_3['answer'][:400]}{'...' if len(result_3['answer']) > 400 else ''}")
print(f"\n  Sources: {[m['chunk_index'] for m in result_3['sources']]}")
print(f"  Scores:  {[f'{s:.2%}' for s in result_3['scores']]}")
print(f"  Time:    {elapsed_3:.2f}s")

# --- Session state ---
print(f"\n{'=' * 60}")
print("SESSION STATE AFTER TESTS")
print(f"{'=' * 60}")
log = session_manager.get_full_log(test_session_rag)
print(f"Total messages: {len(log)}")
for msg in log:
    role_tag = "BOT" if msg["role"] == "assistant" else msg["user_id"]
    preview = msg["content"][:80].replace('\n', ' ')
    print(f"  [{role_tag:>5s}] {preview}{'...' if len(msg['content']) > 80 else ''}")

print(f"\n✓ Dual-context RAG pipeline working — ready for Cell 7 (response throttling)")

TEST: Dual-Context RAG Pipeline

  Session created: 1669032a2048
Conversation context:
  [alice] I've been reading the paper on the emotion detection system.
  [bob] The deep learning architecture looked complex.
  [alice] Yeah, they combined multiple model types together.

TEST 1: Vague message (needs reformulation)

  [bob]: What about the accuracy?
  Reformulated: "Emotion detection system paper accuracy"
  Rewritten:    True

  [bot]:  Based on the paper, the distributed emotion detection system achieved an overall accuracy of 0.96. This high accuracy suggests that the system is effective in identifying both primary and secondary emotions from text inputs. This accuracy underscores the potential of the system for applications requiring nuanced emotional analysis, such as in customer service, healthcare, or social media monitori...

  Sources: [29, 14, 11, 22, 52]
  Scores:  ['83.08%', '81.81%', '81.59%', '80.18%', '79.93%']
  Time:    6.96s

TEST 2: Specific message (no reformulati

In [7]:
# =============================================================================
# CELL 7: Adaptive Response Throttling
# =============================================================================
# Controls when and how the bot speaks. Three systems work together:
#
#   1. Adaptive gap — learns from the bot's own response pattern. Tracks the
#      number of user messages between consecutive bot responses and maintains
#      a rolling average. The minimum gap drifts toward this average, bounded
#      by a floor (MIN_RESPONSE_GAP) and ceiling (MAX_RESPONSE_GAP).
#      If the bot has been spacing responses 6-7 messages apart, it won't
#      suddenly jump in after 2.
#
#   2. Tempo detection — measures time deltas between recent user messages.
#      If messages are arriving rapidly (users in active back-and-forth),
#      the bot holds back even if the gap threshold is met. It waits for
#      a natural pause in the conversation before interjecting.
#      Triggers that fire during rapid exchange are queued, not dropped.
#
#   3. Summary mode — when users signal end of discussion ("let's take a
#      break", "that's it for now"), the bot generates a conversation summary
#      instead of a normal RAG response. Also triggers after extended silence.
#
# Additionally:
#   - Bot ratio check prevents slow-burn domination (unchanged)
#   - Direct @bot always bypasses all throttling (unchanged)
#
# Returns one of four actions:
#   "respond"  — bot should answer now
#   "queue"    — valid trigger, but wait for a pause
#   "throttle" — suppressed (gap/ratio)
#   "summary"  — generate a conversation summary instead of RAG response

# =============================================================================
# CONFIGURATION
# =============================================================================

# Adaptive gap bounds
MIN_RESPONSE_GAP = 2     # Floor — bot always waits at least this many messages
MAX_RESPONSE_GAP = 8     # Ceiling — bot won't wait longer than this
GAP_HISTORY_SIZE = 5     # Number of recent gaps to average for adaptation

# Tempo detection
# If the average time between the last few user messages is below this
# threshold (in seconds), the conversation is "rapid" and the bot holds back.
# For testing in a notebook, use a low value. In production with real users,
# 15-20 seconds would be typical.
RAPID_TEMPO_THRESHOLD = 2.0    # seconds — tune for notebook testing
TEMPO_WINDOW = 3               # number of recent deltas to average

# Summary triggers — phrases that signal end of discussion
SUMMARY_TRIGGER_PHRASES = [
    "let's take a break", "let's wrap up", "that's it for now",
    "we're done", "let's stop here", "good discussion",
    "let's pause", "break time", "that's all", "let's end here",
    "conversation over", "meeting over", "we can stop",
    "summarize", "give us a summary", "sum it up",
    "@bot summarize", "@bot summary"
]

# Summary after extended silence — if no messages for this many seconds,
# and there's meaningful conversation to summarize, offer a recap.
# Set high for testing to avoid accidental triggers.
SILENCE_SUMMARY_THRESHOLD = 300.0  # 5 minutes


class AdaptiveThrottler:
    """
    Tracks bot response patterns and conversation tempo per session.
    Maintains state that the stateless should_throttle() function reads.
    """

    def __init__(self):
        # Per-session tracking
        # gap_history: list of ints — user messages between consecutive bot responses
        # queued_triggers: list of dicts — triggers waiting for a pause
        self.session_data: dict[str, dict] = {}

    def _ensure_session(self, session_id: str):
        """Initialize tracking data for a new session."""
        if session_id not in self.session_data:
            self.session_data[session_id] = {
                "gap_history": [],
                "queued_triggers": []
            }

    def record_bot_response(self, session_id: str, gap: int):
        """
        Record that the bot responded after `gap` user messages.
        Updates the rolling gap history for adaptive threshold calculation.
        """
        self._ensure_session(session_id)
        history = self.session_data[session_id]["gap_history"]
        history.append(gap)
        # Keep only the last GAP_HISTORY_SIZE entries
        if len(history) > GAP_HISTORY_SIZE:
            self.session_data[session_id]["gap_history"] = history[-GAP_HISTORY_SIZE:]

    def get_adaptive_gap(self, session_id: str) -> int:
        """
        Compute the current adaptive minimum gap for this session.

        Logic:
            - If no history yet, use MIN_RESPONSE_GAP (cold start)
            - Otherwise, average the recent gaps and clamp to [floor, ceiling]
            - If the last 3 gaps are trending upward (bot naturally backing off),
              nudge the average up by 1 to maintain momentum
        """
        self._ensure_session(session_id)
        history = self.session_data[session_id]["gap_history"]

        if not history:
            return MIN_RESPONSE_GAP

        avg = sum(history) / len(history)

        # Momentum: if last 3+ gaps are strictly increasing, nudge up
        if len(history) >= 3:
            recent = history[-3:]
            if all(recent[i] < recent[i+1] for i in range(len(recent)-1)):
                avg += 1.0

        # Clamp to bounds
        return max(MIN_RESPONSE_GAP, min(MAX_RESPONSE_GAP, round(avg)))

    def queue_trigger(self, session_id: str, message: str, user_id: str, trigger_type: str):
        """Store a trigger that fired during rapid exchange for later."""
        self._ensure_session(session_id)
        self.session_data[session_id]["queued_triggers"].append({
            "message": message,
            "user_id": user_id,
            "trigger_type": trigger_type,
            "timestamp": time.time()
        })

    def pop_queued_triggers(self, session_id: str) -> list[dict]:
        """Retrieve and clear all queued triggers for a session."""
        self._ensure_session(session_id)
        triggers = self.session_data[session_id]["queued_triggers"]
        self.session_data[session_id]["queued_triggers"] = []
        return triggers

    def has_queued_triggers(self, session_id: str) -> bool:
        """Check if there are triggers waiting for a pause."""
        self._ensure_session(session_id)
        return len(self.session_data[session_id]["queued_triggers"]) > 0

    def get_session_stats(self, session_id: str) -> dict:
        """Return current throttling state for debugging."""
        self._ensure_session(session_id)
        data = self.session_data[session_id]
        return {
            "gap_history": list(data["gap_history"]),
            "adaptive_gap": self.get_adaptive_gap(session_id),
            "queued_triggers": len(data["queued_triggers"])
        }


# --- Initialize the global throttler ---
throttler = AdaptiveThrottler()


def _detect_conversation_tempo(session_id: str) -> dict:
    """
    Measure the tempo of recent conversation by looking at time deltas
    between user messages.

    Returns:
        {
            "is_rapid":   bool,   True if users are in fast back-and-forth
            "avg_delta":  float,  average seconds between recent messages
            "num_deltas": int,    how many deltas were measured
        }
    """
    recent = session_manager.get_recent_messages(session_id, TEMPO_WINDOW + 1)

    # Filter to user messages only (bot messages don't count for tempo)
    user_msgs = [m for m in recent if m["role"] == "user"]

    if len(user_msgs) < 2:
        # Not enough messages to measure tempo
        return {"is_rapid": False, "avg_delta": float("inf"), "num_deltas": 0}

    # Compute deltas between consecutive user messages
    deltas = []
    for i in range(1, len(user_msgs)):
        delta = user_msgs[i]["timestamp"] - user_msgs[i-1]["timestamp"]
        deltas.append(delta)

    avg_delta = sum(deltas) / len(deltas)

    return {
        "is_rapid": avg_delta < RAPID_TEMPO_THRESHOLD,
        "avg_delta": avg_delta,
        "num_deltas": len(deltas)
    }


def _is_summary_request(message: str) -> bool:
    """Check if a message signals the user wants a conversation summary."""
    msg_lower = message.strip().lower()
    for phrase in SUMMARY_TRIGGER_PHRASES:
        if phrase in msg_lower:
            return True
    return False


def generate_summary(session_id: str) -> str:
    """
    Generate a summary of the conversation so far.
    Uses conversation history (not document retrieval) to produce a recap.

    Returns:
        The summary text from the LLM.
    """
    history = session_manager.get_history_for_llm(session_id)

    if len(history) < 2:
        return "There hasn't been enough discussion to summarize yet."

    # Build the conversation text for the summary prompt
    convo_text = "\n".join(
        f"{msg['role'].upper()}: {msg['content']}" for msg in history
    )

    summary_prompt = f"""Here is a team discussion. Provide a concise summary covering:
1. Main topics discussed
2. Key questions raised and whether they were answered
3. Any points of disagreement or uncertainty
4. Open questions still unanswered

Keep it brief — 4-6 sentences maximum.

Conversation:
{convo_text}

Summary:"""

    try:
        response = requests.post(
            f"{OLLAMA_BASE_URL}/api/chat",
            json={
                "model": OLLAMA_MODEL,
                "messages": [
                    {"role": "system", "content": "You are a helpful meeting assistant. Summarize discussions concisely."},
                    {"role": "user", "content": summary_prompt}
                ],
                "stream": False
            },
            timeout=60
        )

        summary = response.json()["message"]["content"].strip()

        # Store the summary as a bot response in the session
        session_manager.add_bot_response(session_id, f"📋 Discussion Summary:\n{summary}")

        return summary

    except Exception as e:
        print(f"  ⚠ Summary generation error: {e}")
        return "Sorry, I couldn't generate a summary right now."


def should_throttle(session_id: str, trigger_type: str, message: str, user_id: str) -> dict:
    """
    Decide what the bot should do with a valid trigger.

    Args:
        session_id:   Active session
        trigger_type: From Cell 4 (DIRECT_TAG bypasses throttling)
        message:      The trigger message (checked for summary requests)
        user_id:      Who sent it

    Returns:
        {
            "action":        "respond" | "queue" | "throttle" | "summary",
            "reason":        str,
            "gap":           int,   user messages since last bot response
            "adaptive_gap":  int,   current adaptive minimum
            "ratio":         float, bot message ratio in recent window
            "tempo":         dict,  tempo detection results
        }
    """
    # --- Summary detection (highest priority after direct tag) ---
    if _is_summary_request(message):
        return {
            "action": "summary",
            "reason": "summary_requested",
            "gap": session_manager.messages_since_last_bot_response(session_id),
            "adaptive_gap": throttler.get_adaptive_gap(session_id),
            "ratio": session_manager.get_bot_message_ratio(session_id),
            "tempo": _detect_conversation_tempo(session_id)
        }

    # --- Direct tag: always respond immediately ---
    if trigger_type == "DIRECT_TAG":
        return {
            "action": "respond",
            "reason": "direct_tag_bypass",
            "gap": session_manager.messages_since_last_bot_response(session_id),
            "adaptive_gap": throttler.get_adaptive_gap(session_id),
            "ratio": session_manager.get_bot_message_ratio(session_id),
            "tempo": _detect_conversation_tempo(session_id)
        }

    gap = session_manager.messages_since_last_bot_response(session_id)
    adaptive_gap = throttler.get_adaptive_gap(session_id)
    ratio = session_manager.get_bot_message_ratio(session_id)
    tempo = _detect_conversation_tempo(session_id)

    # --- Check 1: Adaptive gap ---
    if gap < adaptive_gap:
        return {
            "action": "throttle",
            "reason": f"adaptive_gap ({gap} < {adaptive_gap})",
            "gap": gap,
            "adaptive_gap": adaptive_gap,
            "ratio": ratio,
            "tempo": tempo
        }

    # --- Check 2: Bot ratio ---
    if ratio > MAX_BOT_RATIO:
        return {
            "action": "throttle",
            "reason": f"bot_ratio ({ratio:.0%} > {MAX_BOT_RATIO:.0%})",
            "gap": gap,
            "adaptive_gap": adaptive_gap,
            "ratio": ratio,
            "tempo": tempo
        }

    # --- Check 3: Tempo — queue if conversation is rapid ---
    if tempo["is_rapid"]:
        throttler.queue_trigger(session_id, message, user_id, trigger_type)
        return {
            "action": "queue",
            "reason": f"rapid_tempo (avg {tempo['avg_delta']:.1f}s < {RAPID_TEMPO_THRESHOLD}s)",
            "gap": gap,
            "adaptive_gap": adaptive_gap,
            "ratio": ratio,
            "tempo": tempo
        }

    # --- All checks passed ---
    # If there are queued triggers, the bot should address them too
    # (handled by the orchestrator in Cell 8)
    return {
        "action": "respond",
        "reason": "ok",
        "gap": gap,
        "adaptive_gap": adaptive_gap,
        "ratio": ratio,
        "tempo": tempo
    }


# =============================================================================
# TEST: Verify adaptive throttling
# =============================================================================
print(f"{'=' * 60}")
print("TEST: Adaptive Response Throttling")
print(f"{'=' * 60}")
print(f"  Config: MIN_GAP={MIN_RESPONSE_GAP}, MAX_GAP={MAX_RESPONSE_GAP}, "
      f"RATIO={MAX_BOT_RATIO:.0%}, TEMPO={RAPID_TEMPO_THRESHOLD}s\n")

# --- Test 1: Fresh session — enough messages, should respond ---
print("--- Test 1: Fresh session with enough messages ---")
ts1 = session_manager.create_session()

# Add enough user messages to meet the minimum gap (bot has never spoken,
# so messages_since_last_bot_response returns total user message count)
now = time.time()
for i in range(MIN_RESPONSE_GAP):
    session_manager.add_user_message(ts1, "alice", f"Discussion message {i+1}")
    # Space timestamps apart so tempo detection doesn't flag as rapid
    session_manager.sessions[ts1][-1]["timestamp"] = now + (i * 5.0)

# Now add the trigger message with a gap in time
session_manager.add_user_message(ts1, "alice", "What model did they use?")
session_manager.sessions[ts1][-1]["timestamp"] = now + (MIN_RESPONSE_GAP * 5.0) + 10.0

result = should_throttle(ts1, "QUESTION", "What model did they use?", "alice")
stats = throttler.get_session_stats(ts1)
print(f"  Action: {result['action']}  |  gap: {result['gap']}  |  "
      f"adaptive_gap: {result['adaptive_gap']}  |  reason: {result['reason']}")
print(f"  Throttler state: {stats}")
assert result["action"] == "respond"
print(f"  ✓ Bot responds — enough messages passed, tempo is calm\n")

# --- Test 2: Bot just spoke — should throttle ---
print("--- Test 2: Bot just spoke ---")
session_manager.add_bot_response(ts1, "The paper uses a CNN-LSTM model.")
throttler.record_bot_response(ts1, gap=1)
session_manager.add_user_message(ts1, "bob", "What about accuracy?")

result = should_throttle(ts1, "QUESTION", "What about accuracy?", "bob")
print(f"  Action: {result['action']}  |  gap: {result['gap']}  |  "
      f"adaptive_gap: {result['adaptive_gap']}  |  reason: {result['reason']}")
assert result["action"] == "throttle"
print(f"  ✓ Throttled — only {result['gap']} message(s) since bot spoke\n")

# --- Test 3: Direct @bot bypasses throttle ---
print("--- Test 3: Direct @bot bypasses throttle ---")
result = should_throttle(ts1, "DIRECT_TAG", "@bot what about accuracy?", "bob")
print(f"  Action: {result['action']}  |  reason: {result['reason']}")
assert result["action"] == "respond"
print(f"  ✓ Direct tag bypasses all throttling\n")

# --- Test 4: Adaptive gap increases over time ---
print("--- Test 4: Adaptive gap learning ---")
ts4 = session_manager.create_session()

# Simulate bot responding at increasing intervals
simulated_gaps = [2, 3, 4, 5, 6]
for g in simulated_gaps:
    throttler.record_bot_response(ts4, gap=g)

adaptive = throttler.get_adaptive_gap(ts4)
stats = throttler.get_session_stats(ts4)
print(f"  Gap history:  {stats['gap_history']}")
print(f"  Adaptive gap: {adaptive} (avg={sum(simulated_gaps)/len(simulated_gaps):.1f}, "
      f"momentum nudge applied: {all(simulated_gaps[i] < simulated_gaps[i+1] for i in range(len(simulated_gaps)-1))})")
assert adaptive > MIN_RESPONSE_GAP, "Adaptive gap should have increased"
print(f"  ✓ Gap adapted from {MIN_RESPONSE_GAP} → {adaptive} based on response history\n")

# --- Test 5: Summary request ---
print("--- Test 5: Summary request ---")
ts5 = session_manager.create_session()
session_manager.add_user_message(ts5, "alice", "Good discussion, let's take a break")

result = should_throttle(ts5, "NONE", "Good discussion, let's take a break", "alice")
print(f"  Action: {result['action']}  |  reason: {result['reason']}")
assert result["action"] == "summary"
print(f"  ✓ Summary mode triggered\n")

# --- Test 6: Rapid tempo — queue trigger ---
print("--- Test 6: Rapid conversation tempo ---")
ts6 = session_manager.create_session()

# Simulate rapid-fire messages (timestamps very close together)
now = time.time()
for i in range(5):
    session_manager.add_user_message(ts6, "alice" if i % 2 == 0 else "bob", f"Quick message {i+1}")
    # Backdate timestamps to simulate rapid conversation
    session_manager.sessions[ts6][-1]["timestamp"] = now + (i * 0.5)  # 0.5s apart

# Add a trigger message also in rapid succession
session_manager.add_user_message(ts6, "bob", "What model was used?")
session_manager.sessions[ts6][-1]["timestamp"] = now + 3.0

result = should_throttle(ts6, "QUESTION", "What model was used?", "bob")
tempo = result["tempo"]
print(f"  Action: {result['action']}  |  tempo: avg {tempo['avg_delta']:.1f}s  |  "
      f"rapid: {tempo['is_rapid']}  |  reason: {result['reason']}")

if result["action"] == "queue":
    print(f"  Queued triggers: {throttler.has_queued_triggers(ts6)}")
    print(f"  ✓ Trigger queued — conversation too rapid, will respond at next pause")
else:
    print(f"  ℹ Not queued — tempo {tempo['avg_delta']:.1f}s above threshold "
          f"(messages may not be close enough)")

# --- Test 7: Summary generation ---
print(f"\n--- Test 7: Summary generation ---")
ts7 = session_manager.create_session()
session_manager.add_user_message(ts7, "alice", "The paper covers emotion detection with deep learning.")
session_manager.add_user_message(ts7, "bob", "They used a CNN-LSTM architecture.")
session_manager.add_user_message(ts7, "alice", "What about the accuracy? Did they report numbers?")
session_manager.add_bot_response(ts7, "Based on the paper, the system achieved 85% accuracy on FER2013.")
session_manager.add_user_message(ts7, "bob", "That's decent. What about deployment?")
session_manager.add_user_message(ts7, "alice", "Let's wrap up for now.")

print("  Generating summary...")
summary = generate_summary(ts7)
print(f"\n  Summary:\n  {summary[:400]}{'...' if len(summary) > 400 else ''}")

# --- Final summary ---
print(f"\n{'=' * 60}")
print("THROTTLING SYSTEM SUMMARY")
print(f"{'=' * 60}")
print(f"  ✓ Adaptive gap — learns from bot's response pattern")
print(f"  ✓ Tempo detection — queues triggers during rapid exchange")
print(f"  ✓ Direct @bot — bypasses all throttling")
print(f"  ✓ Summary mode — recaps conversation on request")
print(f"  ✓ Bot ratio — prevents domination over time")
print(f"\n✓ Adaptive throttling ready — proceed to Cell 8 (chat simulation)")

TEST: Adaptive Response Throttling
  Config: MIN_GAP=2, MAX_GAP=8, RATIO=30%, TEMPO=2.0s

--- Test 1: Fresh session with enough messages ---
  Session created: e83e8c6af395
  Action: respond  |  gap: 3  |  adaptive_gap: 2  |  reason: ok
  Throttler state: {'gap_history': [], 'adaptive_gap': 2, 'queued_triggers': 0}
  ✓ Bot responds — enough messages passed, tempo is calm

--- Test 2: Bot just spoke ---
  Action: throttle  |  gap: 1  |  adaptive_gap: 2  |  reason: adaptive_gap (1 < 2)
  ✓ Throttled — only 1 message(s) since bot spoke

--- Test 3: Direct @bot bypasses throttle ---
  Action: respond  |  reason: direct_tag_bypass
  ✓ Direct tag bypasses all throttling

--- Test 4: Adaptive gap learning ---
  Session created: e8000a2b5a34
  Gap history:  [2, 3, 4, 5, 6]
  Adaptive gap: 5 (avg=4.0, momentum nudge applied: True)
  ✓ Gap adapted from 2 → 5 based on response history

--- Test 5: Summary request ---
  Session created: 97a07a2a8529
  Action: summary  |  reason: summary_requested


In [8]:
# =============================================================================
# CELL 8: Chat Simulation and Testing
# =============================================================================
# End-to-end orchestrator that wires together all components:
#   Cell 3 — Multi-user session manager
#   Cell 4 — Trigger detection
#   Cell 5 — Query reformulation
#   Cell 6 — Dual-context RAG pipeline
#   Cell 7 — Adaptive response throttling
#
# The orchestrator processes each message through the full pipeline:
#   message → add to session → trigger? → throttle? → (reformulate → retrieve → generate)
#
# This cell also handles:
#   - Queued trigger recovery (when tempo calms down, pop queued triggers)
#   - Recording bot response gaps for adaptive throttling
#   - Summary generation on break signals
#   - Comprehensive test simulation with a realistic multi-user conversation

# =============================================================================
# ORCHESTRATOR
# =============================================================================

def process_message(
    session_id: str,
    user_id: str,
    message: str,
    timestamp: float = None
) -> dict:
    """
    Main orchestrator — processes a single incoming message through the
    full discussion agent pipeline.

    Flow:
        1. Add message to session
        2. Check for queued triggers (if tempo has calmed)
        3. Run trigger detection
        4. Run throttling
        5. Based on action: respond / queue / throttle / summarize

    Args:
        session_id: Active session
        user_id:    Who sent the message
        message:    Message text
        timestamp:  Optional — override timestamp for testing.
                    If None, uses current time.

    Returns:
        {
            "user_id":        str,
            "message":        str,
            "bot_responded":  bool,
            "bot_response":   str or None,
            "action":         str (respond|queue|throttle|summary|no_trigger),
            "trigger":        dict or None (from Cell 4),
            "throttle":       dict or None (from Cell 7),
            "pipeline":       dict or None (from Cell 6),
            "from_queue":     bool (True if response was from a queued trigger)
        }
    """
    # Step 1: Add the message to the session
    session_manager.add_user_message(session_id, user_id, message)
    if timestamp is not None:
        # Override timestamp for test simulations
        session_manager.sessions[session_id][-1]["timestamp"] = timestamp

    result = {
        "user_id": user_id,
        "message": message,
        "bot_responded": False,
        "bot_response": None,
        "action": "no_trigger",
        "trigger": None,
        "throttle": None,
        "pipeline": None,
        "from_queue": False
    }

    # Step 2: Check for summary request BEFORE trigger detection.
    # Summary phrases like "let's take a break" get classified as non-triggers
    # by the LLM (they're casual chat), but we still want to generate a recap.
    # This check runs first so it doesn't depend on trigger detection.
    if _is_summary_request(message):
        summary = generate_summary(session_id)
        gap = session_manager.messages_since_last_bot_response(session_id)
        throttler.record_bot_response(session_id, gap)

        result["bot_responded"] = True
        result["bot_response"] = summary
        result["action"] = "summary"
        return result

    # Step 3: Check for queued triggers
    # If the tempo has calmed and there are triggers waiting, process the
    # most recent one (they share conversation context, so the latest is
    # most relevant)
    tempo = _detect_conversation_tempo(session_id)
    if not tempo["is_rapid"] and throttler.has_queued_triggers(session_id):
        queued = throttler.pop_queued_triggers(session_id)
        latest = queued[-1]  # Most recent queued trigger

        # Record the gap for adaptive throttling
        gap = session_manager.messages_since_last_bot_response(session_id)

        # Run the RAG pipeline on the queued trigger
        pipeline_result = discussion_rag_query(
            session_id,
            latest["message"],
            latest["user_id"],
            latest["trigger_type"]
        )

        # Record this response for adaptive gap learning
        throttler.record_bot_response(session_id, gap)

        result["bot_responded"] = True
        result["bot_response"] = pipeline_result["answer"]
        result["action"] = "respond"
        result["pipeline"] = pipeline_result
        result["from_queue"] = True
        return result

    # Step 3: Trigger detection
    trigger = detect_trigger(session_id, message, user_id)
    result["trigger"] = trigger

    if not trigger["should_respond"]:
        result["action"] = "no_trigger"
        return result

    # Step 4: Throttling
    throttle = should_throttle(session_id, trigger["trigger_type"], message, user_id)
    result["throttle"] = throttle
    result["action"] = throttle["action"]

    # Step 5: Act based on throttle decision
    if throttle["action"] == "throttle":
        # Suppressed — bot stays silent
        return result

    elif throttle["action"] == "queue":
        # Trigger is valid but conversation is too rapid — queued for later
        return result

    elif throttle["action"] == "summary":
        # Generate conversation summary
        summary = generate_summary(session_id)
        gap = session_manager.messages_since_last_bot_response(session_id)
        throttler.record_bot_response(session_id, gap)

        result["bot_responded"] = True
        result["bot_response"] = summary
        return result

    elif throttle["action"] == "respond":
        # Full RAG pipeline
        gap = session_manager.messages_since_last_bot_response(session_id)

        pipeline_result = discussion_rag_query(
            session_id,
            message,
            user_id,
            trigger["trigger_type"]
        )

        throttler.record_bot_response(session_id, gap)

        result["bot_responded"] = True
        result["bot_response"] = pipeline_result["answer"]
        result["pipeline"] = pipeline_result
        return result

    return result


# =============================================================================
# DISPLAY HELPER
# =============================================================================

def print_message(result: dict, index: int):
    """Pretty-print a processed message result."""
    user = result["user_id"]
    msg = result["message"]

    # User message
    print(f"\n  [{index:2d}] [{user}]: {msg}")

    # Processing details (compact)
    trigger = result.get("trigger")
    throttle = result.get("throttle")

    details = []
    if trigger:
        details.append(f"trigger={trigger['should_respond']}({trigger['trigger_type']})")
        details.append(f"method={trigger['method']}")
    if throttle:
        details.append(f"action={throttle['action']}")
        if throttle.get("adaptive_gap"):
            details.append(f"gap={throttle['gap']}/{throttle['adaptive_gap']}")

    if details:
        print(f"       → {' | '.join(details)}")

    # Bot response
    if result["bot_responded"]:
        queue_tag = " (from queue)" if result["from_queue"] else ""
        pipeline = result.get("pipeline")
        if pipeline and pipeline.get("was_rewritten"):
            print(f"       → reformulated: \"{pipeline['reformulated'][:80]}\"")
        print(f"\n  {'':>5}[bot]{queue_tag}: {result['bot_response'][:300]}"
              f"{'...' if result['bot_response'] and len(result['bot_response']) > 300 else ''}")


# =============================================================================
# SIMULATION: Realistic multi-user discussion
# =============================================================================
print(f"{'=' * 60}")
print("SIMULATION: Multi-User Discussion with Agent")
print(f"{'=' * 60}")
print(f"  Using document: {doc_info['filename']}")
print(f"  Chunks in knowledge base: {collection.count()}\n")

sim_session = session_manager.create_session()

# Realistic conversation — timestamps simulate natural pacing.
# Each tuple: (user_id, message, seconds_after_start)
#
# The conversation has distinct phases:
#   Phase 1: Casual warmup (no triggers expected)
#   Phase 2: Discussion with questions (triggers expected)
#   Phase 3: Rapid back-and-forth (should queue, not interrupt)
#   Phase 4: Pause + direct @bot (should respond + flush queue)
#   Phase 5: Wrap-up with summary request

conversation = [
    # --- Phase 1: Casual warmup ---
    ("alice", "Hey everyone, ready to discuss the paper?",                    0),
    ("bob",   "Yeah, let me pull it up.",                                     5),
    ("alice", "Cool, I thought it was pretty interesting overall.",           12),

    # --- Phase 2: Substantive discussion ---
    ("bob",   "What deep learning model did they use for the main task?",    25),
    ("alice", "I think it was a CNN but I'm not exactly sure.",              40),
    ("bob",   "What datasets did they train on?",                            70),

    # --- Phase 3: Rapid back-and-forth (bot should hold back) ---
    ("alice", "The results section was dense.",                              100),
    ("bob",   "Agreed, lots of numbers.",                                   101),
    ("alice", "What was the accuracy they reported?",                       102),
    ("bob",   "I think around 80-something percent?",                       103),

    # --- Phase 4: Pause, then direct question ---
    ("alice", "Hmm, let me think about that.",                              115),
    ("bob",   "@bot What accuracy did the system achieve?",                 135),

    # --- Phase 5: More discussion, then wrap up ---
    ("alice", "That's helpful, thanks.",                                    155),
    ("bob",   "How was the system deployed?",                               175),
    ("alice", "Good discussion, let's take a break",                        210),
]

print(f"Simulating {len(conversation)} messages across 5 phases...\n")
print(f"{'=' * 60}")

# Track stats
stats = {
    "total_messages": 0,
    "triggers_fired": 0,
    "bot_responses": 0,
    "queued": 0,
    "throttled": 0,
    "summaries": 0,
    "rule_classified": 0,
    "llm_classified": 0
}

for user_id, message, ts_offset in conversation:
    timestamp = time.time() + ts_offset

    result = process_message(sim_session, user_id, message, timestamp=timestamp)
    print_message(result, stats["total_messages"] + 1)

    # Update stats
    stats["total_messages"] += 1
    if result.get("trigger") and result["trigger"]["should_respond"]:
        stats["triggers_fired"] += 1
    if result["bot_responded"]:
        stats["bot_responses"] += 1
    if result["action"] == "queue":
        stats["queued"] += 1
    if result["action"] == "throttle":
        stats["throttled"] += 1
    if result["action"] == "summary":
        stats["summaries"] += 1
    if result.get("trigger"):
        if result["trigger"]["method"] == "rule":
            stats["rule_classified"] += 1
        else:
            stats["llm_classified"] += 1

# =============================================================================
# SIMULATION REPORT
# =============================================================================
print(f"\n\n{'=' * 60}")
print("SIMULATION REPORT")
print(f"{'=' * 60}")
print(f"  Total messages:      {stats['total_messages']}")
print(f"  Triggers fired:      {stats['triggers_fired']}")
print(f"  Bot responses:       {stats['bot_responses']}")
print(f"  Queued (tempo):      {stats['queued']}")
print(f"  Throttled (gap/ratio): {stats['throttled']}")
print(f"  Summaries:           {stats['summaries']}")
print(f"  Rule-classified:     {stats['rule_classified']}")
print(f"  LLM-classified:      {stats['llm_classified']}")

# Adaptive throttler state
throttle_stats = throttler.get_session_stats(sim_session)
print(f"\n  Adaptive throttler:")
print(f"    Gap history:     {throttle_stats['gap_history']}")
print(f"    Current gap:     {throttle_stats['adaptive_gap']}")
print(f"    Queued triggers: {throttle_stats['queued_triggers']}")

# Bot ratio
ratio = session_manager.get_bot_message_ratio(sim_session)
print(f"    Bot ratio:       {ratio:.0%}")

# =============================================================================
# CONVERSATION LOG
# =============================================================================
print(f"\n{'=' * 60}")
print("FULL CONVERSATION LOG")
print(f"{'=' * 60}")

log = session_manager.get_full_log(sim_session)
for i, msg in enumerate(log):
    role_tag = "🤖 BOT" if msg["role"] == "assistant" else f"   {msg['user_id']}"
    content = msg["content"][:120].replace('\n', ' ')
    print(f"  {i+1:2d}. [{role_tag:>8s}] {content}{'...' if len(msg['content']) > 120 else ''}")

# =============================================================================
# VALIDATION CHECKS
# =============================================================================
print(f"\n{'=' * 60}")
print("VALIDATION CHECKS")
print(f"{'=' * 60}")

# Check 1: Bot should NOT respond to casual warmup (Phase 1)
phase1_responses = sum(
    1 for _, msg, ts in conversation[:3]
    if any(m["role"] == "assistant" and abs(m["timestamp"] - (time.time() + ts)) < 1
           for m in log)
)
# Simpler check: count bot responses in first few log entries
early_bot = sum(1 for m in log[:4] if m["role"] == "assistant")
check1 = early_bot == 0
print(f"\n  1. Bot silent during casual warmup: {'✓ PASS' if check1 else '✗ FAIL'}")
print(f"     Bot responses in first 4 messages: {early_bot}")

# Check 2: Bot SHOULD respond to at least one substantive question
check2 = stats["bot_responses"] >= 1
print(f"\n  2. Bot responded to discussion: {'✓ PASS' if check2 else '✗ FAIL'}")
print(f"     Total bot responses: {stats['bot_responses']}")

# Check 3: Direct @bot should always get a response
# Find the @bot message result
direct_responded = False
for user_id, message, ts in conversation:
    if "@bot" in message.lower():
        # Check if the bot responded to this
        for m in log:
            if m["role"] == "assistant" and m["timestamp"] > time.time() + ts - 1:
                direct_responded = True
                break
# Simpler: check if bot_responses > 0 when we know @bot was sent
check3 = stats["bot_responses"] >= 1  # At minimum the @bot got answered
print(f"\n  3. Direct @bot got response: {'✓ PASS' if check3 else '✗ FAIL'}")

# Check 4: Summary was generated
check4 = stats["summaries"] >= 1
print(f"\n  4. Summary generated on wrap-up: {'✓ PASS' if check4 else '✗ FAIL'}")
print(f"     Summaries generated: {stats['summaries']}")

# Check 5: Bot ratio stayed reasonable
check5 = ratio <= 0.5  # Bot should be less than half the conversation
print(f"\n  5. Bot ratio reasonable (≤50%): {'✓ PASS' if check5 else '✗ FAIL'}")
print(f"     Final bot ratio: {ratio:.0%}")

passed = sum([check1, check2, check3, check4, check5])
print(f"\n  Overall: {passed}/5 checks passed")

if passed == 5:
    print(f"\n✓ All checks passed — discussion agent is working end-to-end!")
    print(f"  Ready for FastAPI conversion.")
else:
    print(f"\n⚠ {5 - passed} check(s) failed — review simulation output above")
    print(f"  Note: LLM classification is non-deterministic. Minor variations")
    print(f"  in trigger detection are expected between runs.")

SIMULATION: Multi-User Discussion with Agent
  Using document: sample_doc.pdf
  Chunks in knowledge base: 82

  Session created: e44330369585
Simulating 15 messages across 5 phases...


  [ 1] [alice]: Hey everyone, ready to discuss the paper?
       → trigger=False(NONE) | method=llm

  [ 2] [bob]: Yeah, let me pull it up.
       → trigger=False(NONE) | method=llm

  [ 3] [alice]: Cool, I thought it was pretty interesting overall.
       → trigger=False(NONE) | method=llm

  [ 4] [bob]: What deep learning model did they use for the main task?
       → trigger=True(QUESTION) | method=llm | action=respond | gap=4/2
       → reformulated: "Deep learning model used in main task of discussed paper"

       [bot]:  Based on the document, the deep learning model they used for the main task is RoBERTa. The models were trained, evaluated, and fine-tuned for emotion detection, and this RoBERTa-based model formed the foundation for the subsequent chapters. The team applied several key preprocess

In [9]:
# =============================================================================
# CELL 9: FastAPI Server (In-Notebook)
# =============================================================================
# Runs the full FastAPI app inside the notebook for live testing.
# All functions from Cells 1-8 are already in memory — the app just
# references them directly from the notebook namespace.
#
# Endpoints:
#   GET  /              — Serve the chat frontend (Cell 10's HTML)
#   POST /upload        — Upload and ingest a PDF document
#   GET  /sessions      — List active sessions
#   POST /sessions      — Create a new session
#   DELETE /session/{id} — Delete a session
#   DELETE /collection  — Reset the knowledge base
#   WebSocket /ws/{session_id} — Real-time multi-user chat
#
# Run this cell, then open http://localhost:8000 in your browser.
# To stop the server, run the shutdown cell or restart the kernel.
#
# Prerequisites:
#   pip install fastapi uvicorn python-multipart

import asyncio
import threading
import shutil
from contextlib import asynccontextmanager

import uvicorn
from fastapi import FastAPI, WebSocket, WebSocketDisconnect, UploadFile, File
from fastapi.responses import HTMLResponse, JSONResponse

# =============================================================================
# CONNECTION MANAGER — tracks active WebSocket connections per session
# =============================================================================

class ConnectionManager:
    """
    Manages WebSocket connections grouped by session_id.
    Handles broadcasting messages to all users in a session.
    """

    def __init__(self):
        # {session_id: [(websocket, user_id), ...]}
        self.active_connections: dict[str, list[tuple[WebSocket, str]]] = {}

    async def connect(self, websocket: WebSocket, session_id: str, user_id: str):
        """Accept a WebSocket connection and add it to the session."""
        await websocket.accept()
        if session_id not in self.active_connections:
            self.active_connections[session_id] = []
        self.active_connections[session_id].append((websocket, user_id))

    def disconnect(self, websocket: WebSocket, session_id: str):
        """Remove a WebSocket connection from the session."""
        if session_id in self.active_connections:
            self.active_connections[session_id] = [
                (ws, uid) for ws, uid in self.active_connections[session_id]
                if ws != websocket
            ]
            # Clean up empty sessions
            if not self.active_connections[session_id]:
                del self.active_connections[session_id]

    async def broadcast(self, session_id: str, message: dict):
        """Send a message to all connected users in a session."""
        if session_id not in self.active_connections:
            return
        disconnected = []
        for ws, uid in self.active_connections[session_id]:
            try:
                await ws.send_json(message)
            except Exception:
                disconnected.append(ws)
        # Clean up dead connections
        for ws in disconnected:
            self.disconnect(ws, session_id)

    def get_users(self, session_id: str) -> list[str]:
        """Get list of connected user_ids in a session."""
        if session_id not in self.active_connections:
            return []
        return [uid for _, uid in self.active_connections[session_id]]


manager = ConnectionManager()

# =============================================================================
# FASTAPI APP
# =============================================================================

# Store reference to server thread for shutdown
server_thread = None
server_instance = None

app = FastAPI(title="Discussion Agent", version="0.1.0")


@app.get("/", response_class=HTMLResponse)
async def serve_frontend():
    """Serve the chat frontend HTML."""
    # CHAT_HTML is defined in Cell 10
    try:
        return HTMLResponse(content=CHAT_HTML)
    except NameError:
        return HTMLResponse(
            content="<h1>Run Cell 10 first</h1>"
                    "<p>Cell 10 defines the chat frontend HTML. "
                    "Run it, then refresh this page.</p>"
        )


@app.post("/upload")
async def upload_document(file: UploadFile = File(...)):
    """
    Upload a PDF document for ingestion.
    Saves to UPLOAD_DIR, runs the full ingestion pipeline,
    and returns document metadata.
    """
    global collection, doc_info

    # Validate file type
    if not file.filename.lower().endswith(".pdf"):
        return JSONResponse(
            status_code=400,
            content={"error": "Only PDF files are supported"}
        )

    # Save uploaded file
    file_path = os.path.join(UPLOAD_DIR, file.filename)
    with open(file_path, "wb") as f:
        shutil.copyfileobj(file.file, f)

    try:
        # Run ingestion pipeline (from Cell 2)
        collection, doc_info = upload_and_ingest(file_path)
        return JSONResponse(content={
            "status": "success",
            "document": doc_info
        })
    except Exception as e:
        return JSONResponse(
            status_code=500,
            content={"error": f"Ingestion failed: {str(e)}"}
        )


@app.get("/sessions")
async def list_sessions():
    """List all active sessions with message counts."""
    return JSONResponse(content={
        "sessions": session_manager.list_sessions()
    })


@app.post("/sessions")
async def create_session():
    """Create a new chat session."""
    session_id = session_manager.create_session()
    return JSONResponse(content={
        "session_id": session_id
    })


@app.delete("/session/{session_id}")
async def delete_session(session_id: str):
    """Delete a session and disconnect all users."""
    # Disconnect all WebSocket connections in this session
    if session_id in manager.active_connections:
        for ws, uid in manager.active_connections[session_id]:
            try:
                await ws.close()
            except Exception:
                pass
        del manager.active_connections[session_id]

    session_manager.delete_session(session_id)
    return JSONResponse(content={"status": "deleted", "session_id": session_id})


@app.delete("/collection")
async def reset_collection():
    """Reset the knowledge base by deleting the ChromaDB collection."""
    global collection
    try:
        chroma_client.delete_collection(name=CHROMA_COLLECTION)
        collection = None
        return JSONResponse(content={"status": "collection_reset"})
    except Exception as e:
        return JSONResponse(
            status_code=500,
            content={"error": f"Reset failed: {str(e)}"}
        )


@app.websocket("/ws/{session_id}")
async def websocket_endpoint(websocket: WebSocket, session_id: str):
    """
    WebSocket endpoint for real-time multi-user chat.

    Client sends:
        {"type": "join",    "user_id": "alice"}
        {"type": "message", "user_id": "alice", "content": "Hello"}

    Server broadcasts:
        {"type": "user_message",  "user_id": "...", "content": "..."}
        {"type": "bot_thinking",  "status": "processing"}
        {"type": "bot_response",  "content": "...", "action": "...", "trigger_type": "..."}
        {"type": "bot_summary",   "content": "..."}
        {"type": "user_joined",   "user_id": "...", "users": [...]}
        {"type": "user_left",     "user_id": "...", "users": [...]}
    """
    user_id = None

    try:
        await websocket.accept()
    except Exception as e:
        print(f"  [WS] Accept failed: {e}")
        return

    try:
        # Ensure session exists
        if session_id not in session_manager.sessions:
            session_manager.sessions[session_id] = []

        # Wait for join message
        join_data = await websocket.receive_json()
        if join_data.get("type") != "join" or not join_data.get("user_id"):
            await websocket.close(code=4000, reason="First message must be a join")
            return

        user_id = join_data["user_id"]

        # Track connection (websocket already accepted above)
        if session_id not in manager.active_connections:
            manager.active_connections[session_id] = []
        manager.active_connections[session_id].append((websocket, user_id))

        # Broadcast join notification
        await manager.broadcast(session_id, {
            "type": "user_joined",
            "user_id": user_id,
            "users": manager.get_users(session_id)
        })

        print(f"  [WS] {user_id} joined session {session_id}")

        # Message loop
        while True:
            data = await websocket.receive_json()

            if data.get("type") != "message":
                continue

            content = data.get("content", "").strip()
            if not content:
                continue

            # Broadcast the user's message to all participants
            await manager.broadcast(session_id, {
                "type": "user_message",
                "user_id": user_id,
                "content": content
            })

            # Check if knowledge base exists
            if collection is None or collection.count() == 0:
                # No document uploaded — skip agent processing
                continue

            # Show thinking indicator
            await manager.broadcast(session_id, {
                "type": "bot_thinking",
                "status": "processing"
            })

            # Process through the agent pipeline (runs in thread pool
            # to avoid blocking the event loop during LLM calls)
            result = await asyncio.to_thread(
                process_message, session_id, user_id, content
            )

            # Clear thinking indicator
            await manager.broadcast(session_id, {
                "type": "bot_thinking",
                "status": "idle"
            })

            # If the bot has something to say, broadcast it
            if result["bot_responded"]:
                if result["action"] == "summary":
                    await manager.broadcast(session_id, {
                        "type": "bot_summary",
                        "content": result["bot_response"]
                    })
                else:
                    trigger_type = ""
                    if result.get("trigger"):
                        trigger_type = result["trigger"]["trigger_type"]
                    elif result.get("from_queue"):
                        trigger_type = "QUEUED"

                    await manager.broadcast(session_id, {
                        "type": "bot_response",
                        "content": result["bot_response"],
                        "action": result["action"],
                        "trigger_type": trigger_type,
                        "from_queue": result["from_queue"]
                    })

    except WebSocketDisconnect:
        pass
    except Exception as e:
        print(f"  [WS] Error for {user_id}: {e}")
    finally:
        if user_id:
            manager.disconnect(websocket, session_id)
            await manager.broadcast(session_id, {
                "type": "user_left",
                "user_id": user_id,
                "users": manager.get_users(session_id)
            })
            print(f"  [WS] {user_id} left session {session_id}")


# =============================================================================
# SERVER MANAGEMENT — run in background thread
# =============================================================================

class UvicornServer:
    """Wrapper to run Uvicorn in a background thread with shutdown support."""

    def __init__(self, app, host="0.0.0.0", port=8000):
        self.config = uvicorn.Config(app, host=host, port=port, log_level="warning")
        self.server = uvicorn.Server(self.config)

    def run_in_thread(self):
        """Start the server in a background thread."""
        self.thread = threading.Thread(target=self.server.run, daemon=True)
        self.thread.start()
        # Wait for server to start
        while not self.server.started:
            time.sleep(0.1)
        print(f"  Server started on http://localhost:{self.config.port}")

    def stop(self):
        """Signal the server to shut down."""
        self.server.should_exit = True
        self.thread.join(timeout=5)
        print("  Server stopped")


# --- Start the server ---
print(f"{'=' * 60}")
print("STARTING FASTAPI SERVER")
print(f"{'=' * 60}\n")

# Stop existing server if re-running this cell
try:
    if server_instance is not None:
        print("  Stopping previous server...")
        server_instance.stop()
except Exception:
    pass

server_instance = UvicornServer(app, host="0.0.0.0", port=8000)
server_instance.run_in_thread()

print(f"\n  Endpoints:")
print(f"    GET  http://localhost:8000/          — Chat frontend")
print(f"    POST http://localhost:8000/upload     — Upload PDF")
print(f"    GET  http://localhost:8000/sessions   — List sessions")
print(f"    POST http://localhost:8000/sessions   — Create session")
print(f"    WS   ws://localhost:8000/ws/{{id}}    — Chat WebSocket")
print(f"\n  Document loaded: {'Yes (' + doc_info['filename'] + ')' if doc_info else 'No'}")
print(f"  Knowledge base:  {collection.count() if collection else 0} chunks")
print(f"\n  → Run Cell 10 to define the frontend HTML, then open http://localhost:8000")
print(f"  → To stop the server: server_instance.stop()")

STARTING FASTAPI SERVER

  Server started on http://localhost:8000

  Endpoints:
    GET  http://localhost:8000/          — Chat frontend
    POST http://localhost:8000/upload     — Upload PDF
    GET  http://localhost:8000/sessions   — List sessions
    POST http://localhost:8000/sessions   — Create session
    WS   ws://localhost:8000/ws/{id}    — Chat WebSocket

  Document loaded: Yes (sample_doc.pdf)
  Knowledge base:  82 chunks

  → Run Cell 10 to define the frontend HTML, then open http://localhost:8000
  → To stop the server: server_instance.stop()


  [WS] ab joined session ac121
INGESTION PIPELINE
  Loading PDF: sample_doc.pdf
  Extracted 60,719 chars from 46 pages

  Chunking...
  Quality filter removed 6 low-content chunk(s)
  → 82 chunks (range: 301–799 chars)

  Classification: academic (6 academic signal(s))

  Embedding 82 chunks...


Batches: 100%|██████████| 3/3 [00:00<00:00,  5.62it/s]


  → Embeddings shape: (82, 384)

  Scoring boilerplate...
  → 5 chunk(s) flagged (score > 0.3)

  Ingesting into ChromaDB...
  ⚠ Collection 'rag_documents' has 82 entries — clearing
  Stored batch 0–82

✓ INGESTION COMPLETE
  Doc ID:     6369b3c94190
  File:       sample_doc.pdf
  Type:       academic
  Pages:      46
  Chunks:     82
  Collection: rag_documents (82 entries)
  [WS] ab left session ac121
  [WS] ab1 joined session ac121
  [WS] ab2 joined session ac121
INGESTION PIPELINE
  Loading PDF: sample_doc.pdf
  Extracted 60,719 chars from 46 pages

  Chunking...
  Quality filter removed 6 low-content chunk(s)
  → 82 chunks (range: 301–799 chars)

  Classification: academic (6 academic signal(s))

  Embedding 82 chunks...


Batches: 100%|██████████| 3/3 [00:00<00:00,  6.70it/s]


  → Embeddings shape: (82, 384)

  Scoring boilerplate...
  → 5 chunk(s) flagged (score > 0.3)

  Ingesting into ChromaDB...
  ⚠ Collection 'rag_documents' has 82 entries — clearing
  Stored batch 0–82

✓ INGESTION COMPLETE
  Doc ID:     1693ed4627c5
  File:       sample_doc.pdf
  Type:       academic
  Pages:      46
  Chunks:     82
  Collection: rag_documents (82 entries)
  [WS] ab1 left session ac121
  [WS] ab2 left session ac121


In [10]:
# =============================================================================
# CELL 10: Chat Frontend HTML
# =============================================================================
# Defines the CHAT_HTML string served by GET /
# After running this cell, open http://localhost:8000 in your browser.
#
# Features:
#   - Username entry on join
#   - Session creation / joining
#   - Multi-user real-time chat via WebSocket
#   - Visual distinction: user messages, bot responses, bot summaries
#   - Bot status indicator (monitoring / thinking)
#   - PDF upload area
#   - Connected users list
#   - Auto-scroll with manual override

CHAT_HTML = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Discussion Agent</title>
    <style>
        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }

        :root {
            --bg-primary: #0f1117;
            --bg-secondary: #1a1b26;
            --bg-tertiary: #24253a;
            --bg-input: #1e1f33;
            --border: #2e2f4a;
            --text-primary: #e1e2e8;
            --text-secondary: #8b8da3;
            --text-muted: #5a5c72;
            --accent-blue: #6c8cff;
            --accent-purple: #a78bfa;
            --accent-green: #4ade80;
            --accent-amber: #fbbf24;
            --accent-red: #f87171;
            --bot-bg: #1a2332;
            --bot-border: #2a4a6b;
            --summary-bg: #1a2a1a;
            --summary-border: #2a5a2a;
            --user-bubble: #2a2b45;
            --self-bubble: #1e3a5f;
        }

        body {
            font-family: 'Segoe UI', -apple-system, BlinkMacSystemFont, sans-serif;
            background: var(--bg-primary);
            color: var(--text-primary);
            height: 100vh;
            overflow: hidden;
        }

        /* ============ JOIN SCREEN ============ */
        #join-screen {
            display: flex;
            align-items: center;
            justify-content: center;
            height: 100vh;
            background: var(--bg-primary);
        }

        .join-container {
            background: var(--bg-secondary);
            border: 1px solid var(--border);
            border-radius: 16px;
            padding: 48px;
            width: 420px;
            text-align: center;
        }

        .join-container h1 {
            font-size: 24px;
            font-weight: 600;
            margin-bottom: 8px;
            color: var(--text-primary);
        }

        .join-container p {
            color: var(--text-secondary);
            margin-bottom: 32px;
            font-size: 14px;
            line-height: 1.5;
        }

        .join-container .bot-icon {
            width: 64px;
            height: 64px;
            background: linear-gradient(135deg, var(--accent-blue), var(--accent-purple));
            border-radius: 16px;
            display: flex;
            align-items: center;
            justify-content: center;
            font-size: 28px;
            margin: 0 auto 24px;
        }

        .input-group {
            margin-bottom: 16px;
            text-align: left;
        }

        .input-group label {
            display: block;
            font-size: 13px;
            color: var(--text-secondary);
            margin-bottom: 6px;
            font-weight: 500;
        }

        .input-group input {
            width: 100%;
            padding: 12px 16px;
            background: var(--bg-input);
            border: 1px solid var(--border);
            border-radius: 10px;
            color: var(--text-primary);
            font-size: 15px;
            outline: none;
            transition: border-color 0.2s;
        }

        .input-group input:focus {
            border-color: var(--accent-blue);
        }

        .input-group input::placeholder {
            color: var(--text-muted);
        }

        .join-btn {
            width: 100%;
            padding: 12px;
            background: linear-gradient(135deg, var(--accent-blue), var(--accent-purple));
            border: none;
            border-radius: 10px;
            color: white;
            font-size: 15px;
            font-weight: 600;
            cursor: pointer;
            margin-top: 8px;
            transition: opacity 0.2s;
        }

        .join-btn:hover { opacity: 0.9; }
        .join-btn:disabled { opacity: 0.5; cursor: not-allowed; }

        /* ============ CHAT SCREEN ============ */
        #chat-screen {
            display: none;
            height: 100vh;
            flex-direction: column;
        }

        /* --- Header --- */
        .chat-header {
            display: flex;
            align-items: center;
            justify-content: space-between;
            padding: 16px 24px;
            background: var(--bg-secondary);
            border-bottom: 1px solid var(--border);
            flex-shrink: 0;
        }

        .header-left {
            display: flex;
            align-items: center;
            gap: 12px;
        }

        .header-left .bot-dot {
            width: 36px;
            height: 36px;
            background: linear-gradient(135deg, var(--accent-blue), var(--accent-purple));
            border-radius: 10px;
            display: flex;
            align-items: center;
            justify-content: center;
            font-size: 16px;
        }

        .header-info h2 {
            font-size: 16px;
            font-weight: 600;
        }

        .header-info .session-id {
            font-size: 12px;
            color: var(--text-muted);
            font-family: monospace;
        }

        .header-right {
            display: flex;
            align-items: center;
            gap: 16px;
        }

        .bot-status {
            display: flex;
            align-items: center;
            gap: 6px;
            font-size: 13px;
            color: var(--text-secondary);
        }

        .status-dot {
            width: 8px;
            height: 8px;
            border-radius: 50%;
            background: var(--accent-green);
        }

        .status-dot.thinking {
            background: var(--accent-amber);
            animation: pulse 1s ease-in-out infinite;
        }

        @keyframes pulse {
            0%, 100% { opacity: 1; }
            50% { opacity: 0.3; }
        }

        .users-list {
            display: flex;
            gap: 6px;
            align-items: center;
        }

        .user-chip {
            padding: 4px 10px;
            background: var(--bg-tertiary);
            border-radius: 12px;
            font-size: 12px;
            color: var(--text-secondary);
        }

        .user-chip.self {
            border: 1px solid var(--accent-blue);
            color: var(--accent-blue);
        }

        /* --- Upload Bar --- */
        .upload-bar {
            display: flex;
            align-items: center;
            gap: 12px;
            padding: 10px 24px;
            background: var(--bg-tertiary);
            border-bottom: 1px solid var(--border);
            font-size: 13px;
            flex-shrink: 0;
        }

        .upload-bar .doc-status {
            color: var(--text-secondary);
            flex: 1;
        }

        .upload-bar .doc-status strong {
            color: var(--accent-green);
        }

        .upload-btn {
            padding: 6px 14px;
            background: var(--bg-input);
            border: 1px solid var(--border);
            border-radius: 8px;
            color: var(--text-secondary);
            font-size: 13px;
            cursor: pointer;
            transition: all 0.2s;
        }

        .upload-btn:hover {
            border-color: var(--accent-blue);
            color: var(--accent-blue);
        }

        #file-input { display: none; }

        /* --- Messages Area --- */
        .messages-container {
            flex: 1;
            overflow-y: auto;
            padding: 20px 24px;
            scroll-behavior: smooth;
        }

        .messages-container::-webkit-scrollbar {
            width: 6px;
        }
        .messages-container::-webkit-scrollbar-track {
            background: transparent;
        }
        .messages-container::-webkit-scrollbar-thumb {
            background: var(--border);
            border-radius: 3px;
        }

        .message {
            margin-bottom: 16px;
            max-width: 85%;
            animation: fadeIn 0.2s ease-out;
        }

        @keyframes fadeIn {
            from { opacity: 0; transform: translateY(8px); }
            to { opacity: 1; transform: translateY(0); }
        }

        .message.self {
            margin-left: auto;
        }

        .message .msg-header {
            font-size: 12px;
            color: var(--text-muted);
            margin-bottom: 4px;
            padding-left: 4px;
        }

        .message .msg-header .username {
            font-weight: 600;
            color: var(--text-secondary);
        }

        .message.self .msg-header {
            text-align: right;
            padding-right: 4px;
        }

        .message.self .msg-header .username {
            color: var(--accent-blue);
        }

        .msg-bubble {
            padding: 12px 16px;
            border-radius: 12px;
            font-size: 14px;
            line-height: 1.6;
            word-wrap: break-word;
        }

        .message.user .msg-bubble {
            background: var(--user-bubble);
        }

        .message.self .msg-bubble {
            background: var(--self-bubble);
        }

        /* Bot messages */
        .message.bot {
            max-width: 90%;
        }

        .message.bot .msg-header .username {
            color: var(--accent-purple);
        }

        .message.bot .msg-bubble {
            background: var(--bot-bg);
            border: 1px solid var(--bot-border);
            border-radius: 12px;
        }

        .message.bot .msg-bubble .bot-tag {
            display: inline-block;
            font-size: 10px;
            font-weight: 700;
            text-transform: uppercase;
            letter-spacing: 0.5px;
            padding: 2px 6px;
            border-radius: 4px;
            margin-bottom: 8px;
            background: rgba(108, 140, 255, 0.15);
            color: var(--accent-blue);
        }

        /* Summary messages */
        .message.summary .msg-bubble {
            background: var(--summary-bg);
            border: 1px solid var(--summary-border);
        }

        .message.summary .msg-bubble .bot-tag {
            background: rgba(74, 222, 128, 0.15);
            color: var(--accent-green);
        }

        /* System messages */
        .system-message {
            text-align: center;
            font-size: 12px;
            color: var(--text-muted);
            margin: 12px 0;
            font-style: italic;
        }

        /* Thinking indicator */
        .thinking-indicator {
            display: none;
            margin-bottom: 16px;
            max-width: 90%;
        }

        .thinking-indicator .thinking-bubble {
            display: inline-flex;
            align-items: center;
            gap: 8px;
            padding: 10px 16px;
            background: var(--bot-bg);
            border: 1px solid var(--bot-border);
            border-radius: 12px;
            font-size: 13px;
            color: var(--text-secondary);
        }

        .thinking-dots {
            display: flex;
            gap: 3px;
        }

        .thinking-dots span {
            width: 6px;
            height: 6px;
            background: var(--accent-purple);
            border-radius: 50%;
            animation: bounce 1.2s ease-in-out infinite;
        }

        .thinking-dots span:nth-child(2) { animation-delay: 0.2s; }
        .thinking-dots span:nth-child(3) { animation-delay: 0.4s; }

        @keyframes bounce {
            0%, 60%, 100% { transform: translateY(0); }
            30% { transform: translateY(-6px); }
        }

        /* --- Input Area --- */
        .input-area {
            padding: 16px 24px;
            background: var(--bg-secondary);
            border-top: 1px solid var(--border);
            flex-shrink: 0;
        }

        .input-wrapper {
            display: flex;
            gap: 10px;
            align-items: flex-end;
        }

        .input-wrapper textarea {
            flex: 1;
            padding: 12px 16px;
            background: var(--bg-input);
            border: 1px solid var(--border);
            border-radius: 12px;
            color: var(--text-primary);
            font-size: 14px;
            font-family: inherit;
            resize: none;
            outline: none;
            max-height: 120px;
            min-height: 44px;
            line-height: 1.4;
            transition: border-color 0.2s;
        }

        .input-wrapper textarea:focus {
            border-color: var(--accent-blue);
        }

        .input-wrapper textarea::placeholder {
            color: var(--text-muted);
        }

        .send-btn {
            width: 44px;
            height: 44px;
            background: linear-gradient(135deg, var(--accent-blue), var(--accent-purple));
            border: none;
            border-radius: 12px;
            color: white;
            font-size: 18px;
            cursor: pointer;
            display: flex;
            align-items: center;
            justify-content: center;
            transition: opacity 0.2s;
            flex-shrink: 0;
        }

        .send-btn:hover { opacity: 0.85; }
        .send-btn:disabled { opacity: 0.4; cursor: not-allowed; }

        /* ============ RESPONSIVE ============ */
        @media (max-width: 600px) {
            .chat-header { padding: 12px 16px; }
            .messages-container { padding: 16px; }
            .input-area { padding: 12px 16px; }
            .upload-bar { padding: 8px 16px; }
            .message { max-width: 95%; }
            .join-container { margin: 20px; padding: 32px 24px; }
        }
    </style>
</head>
<body>

<!-- ============ JOIN SCREEN ============ -->
<div id="join-screen">
    <div class="join-container">
        <div class="bot-icon">🤖</div>
        <h1>Discussion Agent</h1>
        <p>Join a collaborative discussion room. The AI agent monitors your conversation and contributes document-backed insights when relevant.</p>
        <div class="input-group">
            <label>Your Name</label>
            <input type="text" id="username-input" placeholder="Enter your name" maxlength="20" autofocus>
        </div>
        <div class="input-group">
            <label>Session ID <span style="color: var(--text-muted)">(leave empty to create new)</span></label>
            <input type="text" id="session-input" placeholder="e.g. abc123">
        </div>
        <button class="join-btn" id="join-btn" onclick="joinChat()">Join Discussion</button>
    </div>
</div>

<!-- ============ CHAT SCREEN ============ -->
<div id="chat-screen">
    <!-- Header -->
    <div class="chat-header">
        <div class="header-left">
            <div class="bot-dot">🤖</div>
            <div class="header-info">
                <h2>Discussion Agent</h2>
                <div class="session-id" id="display-session-id"></div>
            </div>
        </div>
        <div class="header-right">
            <div class="bot-status">
                <div class="status-dot" id="status-dot"></div>
                <span id="status-text">Monitoring</span>
            </div>
            <div class="users-list" id="users-list"></div>
        </div>
    </div>

    <!-- Upload Bar -->
    <div class="upload-bar">
        <div class="doc-status" id="doc-status">No document loaded — upload a PDF to enable the agent</div>
        <input type="file" id="file-input" accept=".pdf" onchange="uploadFile()">
        <button class="upload-btn" onclick="document.getElementById('file-input').click()">Upload PDF</button>
    </div>

    <!-- Messages -->
    <div class="messages-container" id="messages-container">
        <div class="thinking-indicator" id="thinking-indicator">
            <div class="thinking-bubble">
                <div class="thinking-dots">
                    <span></span><span></span><span></span>
                </div>
                Agent is thinking...
            </div>
        </div>
    </div>

    <!-- Input -->
    <div class="input-area">
        <div class="input-wrapper">
            <textarea id="message-input" placeholder="Type a message... (use @bot to directly ask the agent)" rows="1" onkeydown="handleKeyDown(event)"></textarea>
            <button class="send-btn" id="send-btn" onclick="sendMessage()">↑</button>
        </div>
    </div>
</div>

<script>
    // ============ STATE ============
    let ws = null;
    let userId = '';
    let sessionId = '';
    let autoScroll = true;

    // ============ JOIN ============
    async function joinChat() {
        userId = document.getElementById('username-input').value.trim();
        if (!userId) {
            document.getElementById('username-input').focus();
            return;
        }

        sessionId = document.getElementById('session-input').value.trim();

        // Create new session if none specified
        if (!sessionId) {
            try {
                const res = await fetch('/sessions', { method: 'POST' });
                const data = await res.json();
                sessionId = data.session_id;
            } catch (e) {
                alert('Failed to create session: ' + e.message);
                return;
            }
        }

        // Connect WebSocket
        const protocol = location.protocol === 'https:' ? 'wss:' : 'ws:';
        ws = new WebSocket(`${protocol}//${location.host}/ws/${sessionId}`);

        ws.onopen = () => {
            // Send join message
            ws.send(JSON.stringify({ type: 'join', user_id: userId }));

            // Switch to chat screen
            document.getElementById('join-screen').style.display = 'none';
            document.getElementById('chat-screen').style.display = 'flex';
            document.getElementById('display-session-id').textContent = `Session: ${sessionId}`;
            document.getElementById('message-input').focus();

            // Check document status
            checkDocStatus();

            addSystemMessage(`You joined the discussion as ${userId}`);
        };

        ws.onmessage = (event) => {
            const data = JSON.parse(event.data);
            handleMessage(data);
        };

        ws.onclose = () => {
            addSystemMessage('Disconnected from server');
            document.getElementById('send-btn').disabled = true;
        };

        ws.onerror = () => {
            addSystemMessage('Connection error');
        };
    }

    // Allow Enter to join
    document.getElementById('username-input').addEventListener('keydown', (e) => {
        if (e.key === 'Enter') joinChat();
    });
    document.getElementById('session-input').addEventListener('keydown', (e) => {
        if (e.key === 'Enter') joinChat();
    });

    // ============ MESSAGE HANDLING ============
    function handleMessage(data) {
        switch (data.type) {
            case 'user_message':
                addUserMessage(data.user_id, data.content);
                break;
            case 'bot_response':
                addBotMessage(data.content, data.trigger_type, data.from_queue);
                break;
            case 'bot_summary':
                addSummaryMessage(data.content);
                break;
            case 'bot_thinking':
                setThinking(data.status === 'processing');
                break;
            case 'user_joined':
                addSystemMessage(`${data.user_id} joined the discussion`);
                updateUsersList(data.users);
                break;
            case 'user_left':
                addSystemMessage(`${data.user_id} left the discussion`);
                updateUsersList(data.users);
                break;
        }
    }

    // ============ UI BUILDERS ============
    function addUserMessage(sender, content) {
        const container = document.getElementById('messages-container');
        const indicator = document.getElementById('thinking-indicator');
        const isSelf = sender === userId;

        const msgDiv = document.createElement('div');
        msgDiv.className = `message ${isSelf ? 'self' : 'user'}`;

        msgDiv.innerHTML = `
            <div class="msg-header">
                <span class="username">${escapeHtml(sender)}</span>
            </div>
            <div class="msg-bubble">${escapeHtml(content)}</div>
        `;

        container.insertBefore(msgDiv, indicator);
        scrollToBottom();
    }

    function addBotMessage(content, triggerType, fromQueue) {
        const container = document.getElementById('messages-container');
        const indicator = document.getElementById('thinking-indicator');

        let tagLabel = 'Response';
        if (triggerType === 'UNCERTAINTY') tagLabel = 'Clarification';
        else if (triggerType === 'DISAGREEMENT') tagLabel = 'Fact Check';
        else if (triggerType === 'CLARIFICATION') tagLabel = 'Explanation';
        else if (triggerType === 'QUESTION') tagLabel = 'Answer';
        else if (triggerType === 'DIRECT_TAG') tagLabel = 'Direct Reply';
        if (fromQueue) tagLabel += ' (queued)';

        const msgDiv = document.createElement('div');
        msgDiv.className = 'message bot';

        msgDiv.innerHTML = `
            <div class="msg-header">
                <span class="username">🤖 Agent</span>
            </div>
            <div class="msg-bubble">
                <div class="bot-tag">${tagLabel}</div>
                <div>${formatBotContent(content)}</div>
            </div>
        `;

        container.insertBefore(msgDiv, indicator);
        scrollToBottom();
    }

    function addSummaryMessage(content) {
        const container = document.getElementById('messages-container');
        const indicator = document.getElementById('thinking-indicator');

        const msgDiv = document.createElement('div');
        msgDiv.className = 'message bot summary';

        msgDiv.innerHTML = `
            <div class="msg-header">
                <span class="username">🤖 Agent</span>
            </div>
            <div class="msg-bubble">
                <div class="bot-tag">📋 Summary</div>
                <div>${formatBotContent(content)}</div>
            </div>
        `;

        container.insertBefore(msgDiv, indicator);
        scrollToBottom();
    }

    function addSystemMessage(text) {
        const container = document.getElementById('messages-container');
        const indicator = document.getElementById('thinking-indicator');

        const div = document.createElement('div');
        div.className = 'system-message';
        div.textContent = text;

        container.insertBefore(div, indicator);
        scrollToBottom();
    }

    function setThinking(isThinking) {
        const indicator = document.getElementById('thinking-indicator');
        const dot = document.getElementById('status-dot');
        const text = document.getElementById('status-text');

        indicator.style.display = isThinking ? 'block' : 'none';
        dot.className = isThinking ? 'status-dot thinking' : 'status-dot';
        text.textContent = isThinking ? 'Thinking...' : 'Monitoring';

        if (isThinking) scrollToBottom();
    }

    function updateUsersList(users) {
        const container = document.getElementById('users-list');
        container.innerHTML = users.map(u =>
            `<span class="user-chip ${u === userId ? 'self' : ''}">${escapeHtml(u)}</span>`
        ).join('');
    }

    // ============ SEND MESSAGE ============
    function sendMessage() {
        const input = document.getElementById('message-input');
        const content = input.value.trim();
        if (!content || !ws || ws.readyState !== WebSocket.OPEN) return;

        ws.send(JSON.stringify({
            type: 'message',
            user_id: userId,
            content: content
        }));

        input.value = '';
        input.style.height = 'auto';
        input.focus();
    }

    function handleKeyDown(event) {
        if (event.key === 'Enter' && !event.shiftKey) {
            event.preventDefault();
            sendMessage();
        }
        // Auto-resize textarea
        setTimeout(() => {
            const textarea = event.target;
            textarea.style.height = 'auto';
            textarea.style.height = Math.min(textarea.scrollHeight, 120) + 'px';
        }, 0);
    }

    // ============ FILE UPLOAD ============
    async function uploadFile() {
        const fileInput = document.getElementById('file-input');
        const file = fileInput.files[0];
        if (!file) return;

        const docStatus = document.getElementById('doc-status');
        docStatus.innerHTML = `Uploading and processing <strong>${escapeHtml(file.name)}</strong>...`;

        const formData = new FormData();
        formData.append('file', file);

        try {
            const res = await fetch('/upload', { method: 'POST', body: formData });
            const data = await res.json();

            if (data.status === 'success') {
                const info = data.document;
                docStatus.innerHTML = `<strong>${escapeHtml(info.filename)}</strong> — ${info.num_chunks} chunks (${info.doc_type})`;
                addSystemMessage(`Document uploaded: ${info.filename} (${info.num_chunks} chunks)`);
            } else {
                docStatus.innerHTML = `Upload failed: ${data.error}`;
            }
        } catch (e) {
            docStatus.innerHTML = `Upload failed: ${e.message}`;
        }

        fileInput.value = '';
    }

    async function checkDocStatus() {
        // Quick check if a doc is already loaded by trying /sessions
        // The doc_info isn't exposed via API, but we can check collection count
        // For now, just set a generic message — the upload bar will update on upload
        const docStatus = document.getElementById('doc-status');
        docStatus.innerHTML = 'Upload a PDF document to enable the agent, or start chatting if one is already loaded';
    }

    // ============ UTILITIES ============
    function escapeHtml(str) {
        const div = document.createElement('div');
        div.textContent = str;
        return div.innerHTML;
    }

    function formatBotContent(text) {
        // Convert newlines to <br> and basic markdown bold
        return escapeHtml(text)
            .replace(/\\n/g, '<br>')
            .replace(/\\*\\*(.+?)\\*\\*/g, '<strong>$1</strong>');
    }

    function scrollToBottom() {
        if (!autoScroll) return;
        const container = document.getElementById('messages-container');
        setTimeout(() => {
            container.scrollTop = container.scrollHeight;
        }, 50);
    }

    // Detect manual scroll to disable auto-scroll
    document.addEventListener('DOMContentLoaded', () => {
        const container = document.getElementById('messages-container');
        if (container) {
            container.addEventListener('scroll', () => {
                const { scrollTop, scrollHeight, clientHeight } = container;
                autoScroll = scrollHeight - scrollTop - clientHeight < 50;
            });
        }
    });
</script>
</body>
</html>
"""

# =============================================================================
# VERIFY
# =============================================================================
print(f"{'=' * 60}")
print("CHAT FRONTEND READY")
print(f"{'=' * 60}")
print(f"  HTML size:  {len(CHAT_HTML):,} characters")
print(f"  Server:     http://localhost:8000")
print(f"\n  → Open http://localhost:8000 in your browser")
print(f"  → Enter a username and click 'Join Discussion'")
print(f"  → Open a second browser tab with a different username")
print(f"    to simulate multi-user chat")
print(f"\n  Tips:")
print(f"    - Type '@bot' to directly ask the agent")
print(f"    - Say 'let's take a break' to get a summary")
print(f"    - Upload a PDF using the button in the top bar")
print(f"    - The agent monitors silently and interjects when relevant")

CHAT FRONTEND READY
  HTML size:  26,867 characters
  Server:     http://localhost:8000

  → Open http://localhost:8000 in your browser
  → Enter a username and click 'Join Discussion'
  → Open a second browser tab with a different username
    to simulate multi-user chat

  Tips:
    - Type '@bot' to directly ask the agent
    - Say 'let's take a break' to get a summary
    - Upload a PDF using the button in the top bar
    - The agent monitors silently and interjects when relevant
